In [ ]:
_UTILS = None
_UTILS_FAILED = False  
def _load_utils():
    global _UTILS, _UTILS_FAILED
    if _UTILS is not None: return _UTILS
    if _UTILS_FAILED: return None  # don't retry after first failure
    for d in [TASK_DIR, Path('/kaggle/input/competitions/neurogolf-2026')]:
        up = d / 'neurogolf_utils' / 'neurogolf_utils.py'
        if up.exists():
            try:
                spec = importlib.util.spec_from_file_location('ng', up)
                mod  = importlib.util.module_from_spec(spec)
                spec.loader.exec_module(mod)
                _UTILS = mod
                return mod
            except Exception as e:
                print(f'neurogolf_utils load failed: {e}')
                _UTILS_FAILED = True
                return None
    _UTILS_FAILED = True
    return None

def exact_cost_via_utils(model):
    utils = _load_utils()
    if utils is None: return None
    fname = None
    try:
        with tempfile.NamedTemporaryFile(suffix='.onnx', delete=False) as f:
            f.write(model.SerializeToString()); f.flush(); fname = f.name
        macs, mem, params = utils.score_network(fname)
        return int(macs + mem + params)
    except Exception:
        return None
    finally:
        if fname:
            try: os.unlink(fname)
            except: pass

def load_task(path):
    with open(path) as f: return json.load(f)

def grid_to_tensor(grid):
    g = np.array(grid, dtype=np.int32)
    h, w = g.shape
    t = np.zeros((1, C, H, W), dtype=np.float32)
    if h > H or w > W: return t
    for v in range(C):
        t[0, v, :h, :w] = (g == v).astype(np.float32)
    return t

def tensor_to_grid(t, out_h, out_w):
    arr = t[0]
    out_h = min(out_h, H); out_w = min(out_w, W)
    slice_ = arr[:, :out_h, :out_w]
    return np.where(slice_.max(0) < 0.5, 0, slice_.argmax(0)).astype(int).tolist()

def shapes_match(p):
    return np.array(p['input']).shape == np.array(p['output']).shape

In [ ]:
import os
os.system('pip install -q onnx==1.21.0 onnxruntime==1.24.4 onnx-tool==1.0.1')

import os, json, zipfile, math, copy, sys, time, traceback, random, tempfile, gc
import importlib.util
import numpy as np
from pathlib import Path
from itertools import product as iprod
import re, io
from collections import Counter

import onnx
import onnx.helper as oh
import onnx.numpy_helper as onh
from onnx import TensorProto

import onnxruntime as ort
import onnx_tool

sys.path.insert(0, '/kaggle/input/competitions/neurogolf-2026/neurogolf_utils')
try:
    import neurogolf_utils as nu
    print('neurogolf_utils loaded OK')
    NU_AVAILABLE = True
except Exception as e:
    print(f'neurogolf_utils NOT available: {e}')
    NU_AVAILABLE = False

ort.set_default_logger_severity(3)

# ---- CONFIG ----
TASK_DIR   = Path('/kaggle/input/competitions/neurogolf-2026')
OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(exist_ok=True)

C, H, W = 10, 30, 30
HW  = H * W
CHW = C * H * W

MAX_ARC_GEN_VALIDATE = 9999
NUM_TASKS = 400
MAX_BYTES = int(1.44 * 1024 * 1024)

BANNED_OPS = {'Loop', 'Scan', 'NonZero', 'Unique', 'If', 'Function'}
TASK_PATTERN = re.compile(r'^task\d{3}\.onnx$')

EXCLUDED_TASKS = {21, 55, 80, 184, 202, 366}

SOLUTION_DIRS = {
    'aliafzal_tiny':     Path('/kaggle/input/notebooks/aliafzal9323/neurogolf-2026-tiny-onnx-solver'),
    'artem_logic2':      Path('/kaggle/input/notebooks/artemnazemtsev/neurogolf-logic-driven-ensembling-part-2'),
    'artem_logic4':      Path('/kaggle/input/notebooks/artemnazemtsev/neurogolf-logic-driven-ensembling-part-4'),
    'jonathanchan':      Path('/kaggle/input/notebooks/jonathanchan/ngc26-constraint-smart-logic-mix-blending'),
    'needless090':       Path('/kaggle/input/notebooks/needless090/neurogolf-4250'),
    'konbu17_5344':      Path('/kaggle/input/notebooks/konbu17/neurogolf-2026-blended-401-tasks-lb-5344'),
    'beicicc_6233':      Path('/kaggle/input/notebooks/beicicc/neurogolf-6233-36-public-score-open-solution'),
    'afr1ste_5501':      Path('/kaggle/input/datasets/afr1ste/neurogolf-5501-78-open-submission-artifact'),
    'konbu17_v117':      Path('/kaggle/input/datasets/konbu17/neurogolf-2026-blended-401-v117'),
    'jonathanchan_v3':   Path('/kaggle/input/datasets/jonathanchan/ngc26-public-v3/submission'),
    'op37_artifact':     Path('/kaggle/input/notebooks/afr1ste/neurogolf-6285-95-public-score-open-solution'),
    'artemn':            Path('/kaggle/input/notebooks/artemnazemtsev/neurogolf-acking-multiple-tasks-part-3'),
    'artem_update':      Path('/kaggle/input/notebooks/artemnazemtsev/neurogolf-update-6500-hacking-part-2'),
}



def load_graphs_from_dir(directory, label):
    graphs = {}
    if not directory.exists():
        print(f'  [{label}] not found: {directory}')
        return graphs

    for fpath in directory.rglob('*'):
        if fpath.is_file():
            if TASK_PATTERN.match(fpath.name):
                graphs[fpath.name] = fpath.read_bytes()

            elif fpath.suffix == '.zip':
                try:
                    with zipfile.ZipFile(fpath) as z:
                        found = 0
                        for entry in z.namelist():
                            bn = os.path.basename(entry)
                            if TASK_PATTERN.match(bn):
                                graphs.setdefault(bn, z.read(entry))
                                found += 1
                        if found:
                            print(f'  [{label}] {found} files from {fpath.name}')
                except Exception as e:
                    print(f'  [{label}] zip error: {e}')

    return graphs


all_source_models = {}  # fname -> list of (label, raw_bytes)

for label, dir_path in SOLUTION_DIRS.items():
    graphs = load_graphs_from_dir(dir_path, label)
    print(f'[{label}] {len(graphs)} models')

    for fname, raw in graphs.items():
        all_source_models.setdefault(fname, []).append((label, raw))

print(f'\nUnique tasks across all sources: {len(all_source_models)}')

floor_solutions = {}
floor_costs     = {}

In [ ]:
floor_backups = {}

def check_model_correct(model, pairs, timeout_pairs=40):
    """Validate model. Uses GPU if available. Caps at timeout_pairs for speed."""
    try:
        buf = model.SerializeToString()
        so = ort.SessionOptions(); so.log_severity_level = 3
        sess = ort.InferenceSession(
            buf, sess_options=so,
            providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
        )
        for p in pairs[:timeout_pairs]:
            inp = grid_to_tensor(p['input'])
            og = np.array(p['output'])
            oh_, ow_ = og.shape
            pred = sess.run(None, {'input': inp})[0]
            pred_grid = tensor_to_grid(pred, oh_, ow_)
            if pred_grid != p['output']:
                return False
        return True
    except Exception:
        return False


def estimate_model_cost(model):
    """Uses neurogolf_utils if available (exact match to organizer scoring).
    Fallback: params + memory_bytes + Conv/Gemm/MatMul MACs only.
    NOTE: Mul, Add, Relu, Gather, Tile, Pad, Resize, Transpose, Slice → 0 MACs."""
    ec = exact_cost_via_utils(model)
    if ec is not None:
        return ec

    try:
        total_params = 0; total_bytes = 0
        tensors = {}

        for init in model.graph.initializer:
            arr = onh.to_array(init); tensors[init.name] = arr
            total_params += arr.size; total_bytes += arr.nbytes

        for node in model.graph.node:
            if node.op_type == 'Constant':
                for attr in node.attribute:
                    if attr.HasField('t'):
                        try:
                            arr = onh.to_array(attr.t)
                            if node.output:
                                tensors[node.output[0]] = arr
                            total_params += arr.size; total_bytes += arr.nbytes
                        except Exception:
                            pass

        total_macs = 0
        for node in model.graph.node:
            if node.op_type == 'Conv':
                if len(node.input) >= 2 and node.input[1] in tensors:
                    w = tensors[node.input[1]]
                    if w.ndim == 4:
                        c_out, c_in, kh, kw = w.shape
                        total_macs += c_out * c_in * kh * kw * H * W
            elif node.op_type in ('Gemm', 'MatMul'):
                if len(node.input) >= 2 and node.input[1] in tensors:
                    w = tensors[node.input[1]]
                    if w.ndim == 2:
                        total_macs += w.shape[0] * w.shape[1]

        return total_params + total_bytes + total_macs
    except Exception:
        return float('inf')

SAFE_SCRUB_TASKS = {
    15, 24, 25, 50, 63, 67, 69, 73, 81, 85, 86, 92, 98, 125, 127, 132,
    157, 160, 161, 162, 166, 180, 187, 193, 196, 198, 220, 222, 224, 225,
    226, 230, 232, 254, 278, 293, 298, 299, 303, 314, 320, 323, 340, 344, 346, 389
}

BAD_SCRUB_TASKS = {4, 17, 34, 64, 66, 74, 119, 192, 231, 312, 329, 359, 376}


def dim_scrub(raw):
    try:
        m = onnx.load_model_from_string(raw)
        g = m.graph

        for vi in [g.input[0], g.output[0]]:
            shape = vi.type.tensor_type.shape
            for i in [2, 3]:
                if i < len(shape.dim):
                    shape.dim[i].ClearField('dim_value')
                    shape.dim[i].ClearField('dim_param')

        return m.SerializeToString()

    except Exception:
        return raw


def validate_official(task_id_int, raw):
    """Use neurogolf_utils for exact validation + cost."""
    if not NU_AVAILABLE:
        return None
    try:
        if len(raw) > MAX_BYTES:
            return None
        onnx.load_model_from_string(raw)
    except Exception:
        return None

    try:
        sess = ort.InferenceSession(raw, providers=['CPUExecutionProvider'])
        examples = nu.load_examples(task_id_int)
        agi_pass, agi_fail, _ = nu.verify_subset(sess, examples['train'] + examples['test'])
        gen_pass, gen_fail, _ = nu.verify_subset(sess, examples['arc-gen'][:20])

        if agi_fail > 0 or gen_fail > 0:
            return None

        tmp = f'/tmp/vfloor{task_id_int:03d}.onnx'
        with open(tmp, 'wb') as f:
            f.write(raw)

        macs, mem, params = nu.score_network(tmp)
        if None in (macs, mem, params):
            return None

        cost = int(macs + mem + params)
        return {'cost': cost, 'score': max(1.0, 25.0 - math.log(cost))}

    except Exception:
        return None
    finally:
        try:
            del sess
        except Exception:
            pass
        gc.collect()
        gc.collect()


def _static_ok(raw):
    """Size, banned ops, static shapes (April 28 update)."""
    if len(raw) > MAX_BYTES:
        return False

    try:
        m = onnx.load_model_from_string(raw)
    except Exception:
        return False

    ops = {nd.op_type for nd in m.graph.node}
    if BANNED_OPS & ops:
        return False

    # Reject dynamic/missing shapes — April 28 grader requirement
    try:
        m_inf = onnx.shape_inference.infer_shapes(m)
        for vi in m_inf.graph.value_info:
            tt = vi.type.tensor_type
            if not tt.HasField('shape'):
                return False
            for dim in tt.shape.dim:
                if dim.dim_param or dim.dim_value <= 0:
                    return False
    except Exception:
        return False

    return True


def _validate_floor_fallback(raw, task_data):
    """Fallback when neurogolf_utils unavailable."""
    try:
        opts = ort.SessionOptions(); opts.log_severity_level = 3
        sess = ort.InferenceSession(
            raw, sess_options=opts,
            providers=['CPUExecutionProvider']
        )
        inp_name = sess.get_inputs()[0].name

        for split in ('train', 'test', 'arc-gen'):
            for pair in task_data.get(split, []):
                inp = grid_to_tensor(pair['input'])
                tgt = np.array(pair['output'])
                th, tw = tgt.shape
                out = sess.run(None, {inp_name: inp})[0]
                pred = np.argmax(out[0, :, :th, :tw], axis=0)
                if not np.array_equal(pred, tgt):
                    return False

        return True
    except Exception:
        return False


# Load task JSONs (used by fallback only)
task_jsons = {}

if TASK_DIR.exists():
    for jp in TASK_DIR.rglob('*.json'):
        key = jp.stem + '.onnx'
        if TASK_PATTERN.match(key):
            try:
                task_jsons[key] = json.loads(jp.read_text())
            except Exception:
                pass

print(f'Loaded {len(task_jsons)} task JSONs')

print('Selecting cheapest valid floor model per task...', flush=True)
rej_counts = Counter()

PRIORITY_SOURCES = {
    'konbu17_v117': 0,   
    'beicicc_6233':  0,   
    'artemn':  0,
    'afr1ste_5501': 1,
    'needless090':  1,
    'konbu17_5344': 1,
    'konbu17_5344_ds': 1,
    'artem_logic4': 1,   
    'afr1ste':      2,
    'konbu17_v84':  2,
    'artem_logic3': 3,
    'artem_4275':   4,
    'artem_gamble': 4,
    'jonathanchan': 4,
}

for fname, candidates in all_source_models.items():
    m = re.match(r'task(\d{3})\.onnx', fname)
    if not m:
        continue

    task_id_int = int(m.group(1))


    sorted_cands = sorted(
        candidates,
        key=lambda x: (PRIORITY_SOURCES.get(x[0], 4), len(x[1]))
    )

    passing = []

    for label, raw in sorted_cands:
        if _static_ok(raw):
            passing.append(raw)
            if len(passing) >= 10:
                break
        else:
            rej_counts[f'{label}-static'] += 1

    if passing:
        floor_solutions[fname] = passing[0]
        floor_costs[fname] = len(passing[0])

        if len(passing) > 1:
            floor_backups[fname] = passing[1:]

    gc.collect()

print(f'Floor solutions selected: {len(floor_solutions)}')

if rej_counts:
    print(f'Rejections: {dict(rej_counts)}')

if floor_costs:
    vals = [v for v in floor_costs.values() if v != float('inf')]
    if vals:
        print(f'Floor cost — min: {min(vals)}, max: {max(vals)}, mean: {int(sum(vals)/len(vals))}')

In [ ]:
def make_identity_onnx():
    X = oh.make_tensor_value_info('input', TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info('output', TensorProto.FLOAT, [1, C, H, W])
    node = oh.make_node('Identity', ['input'], ['output'])
    graph = oh.make_graph([node], 'identity', [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid('', 17)])
    model.ir_version = 8
    return model

def _const_node(name, arr):
    t = onh.from_array(arr, name=name + '_v')
    return oh.make_node('Constant', [], [name], value=t)

def _slice_reverse_nodes(axes, dims, in_name, out_name, suf=''):
    starts = [d - 1 for d in dims]
    ends   = [-d - 1 for d in dims]
    steps  = [-1] * len(dims)
    return [
        _const_node(f'rs_s{suf}', np.asarray(starts, dtype=np.int64)),
        _const_node(f'rs_e{suf}', np.asarray(ends,   dtype=np.int64)),
        _const_node(f'rs_a{suf}', np.asarray(axes,   dtype=np.int64)),
        _const_node(f'rs_p{suf}', np.asarray(steps,  dtype=np.int64)),
        oh.make_node('Slice',
                     [in_name, f'rs_s{suf}', f'rs_e{suf}', f'rs_a{suf}', f'rs_p{suf}'],
                     [out_name]),
    ]

def _pad_to_canvas_nodes(in_name, out_name, out_h, out_w, suf=''):
    pads = np.asarray([0, 0, 0, 0, 0, 0, H - out_h, W - out_w], dtype=np.int64)
    return [
        _const_node(f'pd{suf}', pads),
        oh.make_node('Pad', [in_name, f'pd{suf}'], [out_name], mode='constant'),
    ]

def _crop_nodes(in_name, out_name, h, w, suf=''):
    return [
        _const_node(f'cs{suf}', np.asarray([0, 0], dtype=np.int64)),
        _const_node(f'ce{suf}', np.asarray([h, w], dtype=np.int64)),
        _const_node(f'ca{suf}', np.asarray([2, 3], dtype=np.int64)),
        oh.make_node('Slice', [in_name, f'cs{suf}', f'ce{suf}', f'ca{suf}'], [out_name]),
    ]

def _build_model_from_nodes(nodes, name):
    X = oh.make_tensor_value_info('input',  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info('output', TensorProto.FLOAT, [1, C, H, W])
    graph = oh.make_graph(nodes, name, [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid('', 17)])
    model.ir_version = 8
    return model

def make_cheap_rot180_framed(h, w):
    nodes  = _crop_nodes('input', 'content', h, w, '0')
    nodes += _slice_reverse_nodes([2, 3], [h, w], 'content', 'rotated', 'r')
    nodes += _pad_to_canvas_nodes('rotated', 'output', h, w, 'p')
    return _build_model_from_nodes(nodes, 'rot180_framed')

def make_cheap_rot90cw_framed(h, w):
    nodes  = _crop_nodes('input', 'content', h, w, '0')
    nodes += [oh.make_node('Transpose', ['content'], ['t'], perm=[0, 1, 3, 2])]
    nodes += _slice_reverse_nodes([3], [h], 't', 'rotated', 'r')
    nodes += _pad_to_canvas_nodes('rotated', 'output', w, h, 'p')
    return _build_model_from_nodes(nodes, 'rot90cw_framed')

def make_cheap_rot90ccw_framed(h, w):
    nodes  = _crop_nodes('input', 'content', h, w, '0')
    nodes += [oh.make_node('Transpose', ['content'], ['t'], perm=[0, 1, 3, 2])]
    nodes += _slice_reverse_nodes([2], [w], 't', 'rotated', 'r')
    nodes += _pad_to_canvas_nodes('rotated', 'output', w, h, 'p')
    return _build_model_from_nodes(nodes, 'rot90ccw_framed')

def make_cheap_flip_h_framed(h, w):
    nodes  = _crop_nodes('input', 'content', h, w, '0')
    nodes += _slice_reverse_nodes([2], [h], 'content', 'flipped', 'r')
    nodes += _pad_to_canvas_nodes('flipped', 'output', h, w, 'p')
    return _build_model_from_nodes(nodes, 'flip_h_framed')

def make_cheap_flip_w_framed(h, w):
    nodes  = _crop_nodes('input', 'content', h, w, '0')
    nodes += _slice_reverse_nodes([3], [w], 'content', 'flipped', 'r')
    nodes += _pad_to_canvas_nodes('flipped', 'output', h, w, 'p')
    return _build_model_from_nodes(nodes, 'flip_w_framed')

def make_cheap_transpose_framed(h, w):
    nodes  = _crop_nodes('input', 'content', h, w, '0')
    nodes += [oh.make_node('Transpose', ['content'], ['t'], perm=[0, 1, 3, 2])]
    nodes += _pad_to_canvas_nodes('t', 'output', w, h, 'p')
    return _build_model_from_nodes(nodes, 'transpose_framed')

def make_cheap_anti_transpose_framed(h, w):
    nodes  = _crop_nodes('input', 'content', h, w, '0')
    nodes += [oh.make_node('Transpose', ['content'], ['t'], perm=[0, 1, 3, 2])]
    nodes += _slice_reverse_nodes([2, 3], [w, h], 't', 'rotated', 'r')
    nodes += _pad_to_canvas_nodes('rotated', 'output', w, h, 'p')
    return _build_model_from_nodes(nodes, 'anti_transpose_framed')

def make_cheap_crop(r0, c0, out_h, out_w):
    nodes = [
        _const_node('cs', np.asarray([r0, c0], dtype=np.int64)),
        _const_node('ce', np.asarray([r0 + out_h, c0 + out_w], dtype=np.int64)),
        _const_node('ca', np.asarray([2, 3], dtype=np.int64)),
        oh.make_node('Slice', ['input', 'cs', 'ce', 'ca'], ['cropped']),
    ]
    nodes += _pad_to_canvas_nodes('cropped', 'output', out_h, out_w, 'p')
    return _build_model_from_nodes(nodes, 'crop_net')

def make_cheap_tile(tr, tc, in_h, in_w):
    nodes  = _crop_nodes('input', 'content', in_h, in_w, '0')
    nodes += [
        _const_node('tile_r', np.asarray([1, 1, tr, tc], dtype=np.int64)),
        oh.make_node('Tile', ['content', 'tile_r'], ['tiled']),
    ]
    nodes += _pad_to_canvas_nodes('tiled', 'output', in_h * tr, in_w * tc, 'p')
    return _build_model_from_nodes(nodes, 'tile_net')

def make_cheap_upscale(f, in_h, in_w):
    nodes  = _crop_nodes('input', 'content', in_h, in_w, '0')
    nodes += [
        _const_node('rsz_roi',    np.asarray([], dtype=np.float32)),
        _const_node('rsz_scales', np.asarray([], dtype=np.float32)),
        _const_node('rsz_sizes',  np.asarray([1, C, in_h * f, in_w * f], dtype=np.int64)),
        oh.make_node('Resize', ['content', 'rsz_roi', 'rsz_scales', 'rsz_sizes'], ['upscaled'],
                     mode='nearest', coordinate_transformation_mode='asymmetric',
                     nearest_mode='floor'),
    ]
    nodes += _pad_to_canvas_nodes('upscaled', 'output', in_h * f, in_w * f, 'p')
    return _build_model_from_nodes(nodes, 'upscale_net')

def make_cheap_concat_flip_w(in_h, in_w):
    nodes  = _crop_nodes('input', 'content', in_h, in_w, '0')
    nodes += _slice_reverse_nodes([3], [in_w], 'content', 'flipped', 'fw')
    nodes += [oh.make_node('Concat', ['content', 'flipped'], ['concated'], axis=3)]
    nodes += _pad_to_canvas_nodes('concated', 'output', in_h, in_w * 2, 'p')
    return _build_model_from_nodes(nodes, 'concat_flip_w')

def make_cheap_concat_flip_h(in_h, in_w):
    nodes  = _crop_nodes('input', 'content', in_h, in_w, '0')
    nodes += _slice_reverse_nodes([2], [in_h], 'content', 'flipped', 'fh')
    nodes += [oh.make_node('Concat', ['content', 'flipped'], ['concated'], axis=2)]
    nodes += _pad_to_canvas_nodes('concated', 'output', in_h * 2, in_w, 'p')
    return _build_model_from_nodes(nodes, 'concat_flip_h')

def make_cheap_quadrant_mirror(in_h, in_w):
    nodes  = _crop_nodes('input', 'content', in_h, in_w, '0')
    nodes += _slice_reverse_nodes([3], [in_w], 'content', 'fw', 'fw')
    nodes += [oh.make_node('Concat', ['content', 'fw'], ['top_row'], axis=3)]
    nodes += _slice_reverse_nodes([2], [in_h], 'content', 'fh', 'fh')
    nodes += _slice_reverse_nodes([2, 3], [in_h, in_w], 'content', 'fr', 'fr')
    nodes += [oh.make_node('Concat', ['fh', 'fr'], ['bot_row'], axis=3)]
    nodes += [oh.make_node('Concat', ['top_row', 'bot_row'], ['full'], axis=2)]
    nodes += _pad_to_canvas_nodes('full', 'output', in_h * 2, in_w * 2, 'p')
    return _build_model_from_nodes(nodes, 'quadrant_mirror')

def make_cheap_self_concat_h(in_h, in_w):
    nodes  = _crop_nodes('input', 'content', in_h, in_w, '0')
    nodes += [oh.make_node('Concat', ['content', 'content'], ['concated'], axis=2)]
    nodes += _pad_to_canvas_nodes('concated', 'output', in_h * 2, in_w, 'p')
    return _build_model_from_nodes(nodes, 'self_concat_h')

def make_cheap_self_concat_w(in_h, in_w):
    nodes  = _crop_nodes('input', 'content', in_h, in_w, '0')
    nodes += [oh.make_node('Concat', ['content', 'content'], ['concated'], axis=3)]
    nodes += _pad_to_canvas_nodes('concated', 'output', in_h, in_w * 2, 'p')
    return _build_model_from_nodes(nodes, 'self_concat_w')

def make_cheap_framed_translation(grid_h, grid_w, dr, dc):
    src_r0 = max(0, -dr); src_r1 = min(grid_h, grid_h - dr)
    src_c0 = max(0, -dc); src_c1 = min(grid_w, grid_w - dc)
    dst_r0 = max(0, dr);  dst_c0 = max(0, dc)
    rh = src_r1 - src_r0; rw = src_c1 - src_c0
    if rh <= 0 or rw <= 0: return None
    nodes = [
        _const_node('fts', np.asarray([src_r0, src_c0], dtype=np.int64)),
        _const_node('fte', np.asarray([src_r1, src_c1], dtype=np.int64)),
        _const_node('fta', np.asarray([2, 3],           dtype=np.int64)),
        oh.make_node('Slice', ['input', 'fts', 'fte', 'fta'], ['fsliced']),
    ]
    pads = np.asarray([0, 0, dst_r0, dst_c0, 0, 0, H - dst_r0 - rh, W - dst_c0 - rw], dtype=np.int64)
    nodes += [_const_node('fpp', pads), oh.make_node('Pad', ['fsliced', 'fpp'], ['output'], mode='constant')]
    return _build_model_from_nodes(nodes, 'framed_translate')

def make_cheap_downscale(f, in_h, in_w):
    out_h = in_h // f; out_w = in_w // f
    nodes = [
        _const_node('ds_s', np.asarray([0, 0],       dtype=np.int64)),
        _const_node('ds_e', np.asarray([in_h, in_w], dtype=np.int64)),
        _const_node('ds_a', np.asarray([2, 3],       dtype=np.int64)),
        _const_node('ds_p', np.asarray([f, f],       dtype=np.int64)),
        oh.make_node('Slice', ['input', 'ds_s', 'ds_e', 'ds_a', 'ds_p'], ['downsampled']),
    ]
    nodes += _pad_to_canvas_nodes('downsampled', 'output', out_h, out_w, 'p')
    return _build_model_from_nodes(nodes, 'downscale')

def make_cheap_row_broadcast(in_h, in_w, row_idx=0):
    nodes = [
        _const_node('rb_s', np.asarray([row_idx],     dtype=np.int64)),
        _const_node('rb_e', np.asarray([row_idx + 1], dtype=np.int64)),
        _const_node('rb_a', np.asarray([2],           dtype=np.int64)),
        oh.make_node('Slice', ['input', 'rb_s', 'rb_e', 'rb_a'], ['one_row']),
        _const_node('rb_r', np.asarray([1, 1, in_h, 1], dtype=np.int64)),
        oh.make_node('Tile', ['one_row', 'rb_r'], ['tiled']),
        _const_node('rb_cs', np.asarray([0, 0],       dtype=np.int64)),
        _const_node('rb_ce', np.asarray([in_h, in_w], dtype=np.int64)),
        _const_node('rb_ca', np.asarray([2, 3],       dtype=np.int64)),
        oh.make_node('Slice', ['tiled', 'rb_cs', 'rb_ce', 'rb_ca'], ['cropped']),
    ]
    nodes += _pad_to_canvas_nodes('cropped', 'output', in_h, in_w, 'p')
    return _build_model_from_nodes(nodes, 'row_broadcast')

def make_cheap_col_broadcast(in_h, in_w, col_idx=0):
    nodes = [
        _const_node('cb_s', np.asarray([col_idx],     dtype=np.int64)),
        _const_node('cb_e', np.asarray([col_idx + 1], dtype=np.int64)),
        _const_node('cb_a', np.asarray([3],           dtype=np.int64)),
        oh.make_node('Slice', ['input', 'cb_s', 'cb_e', 'cb_a'], ['one_col']),
        _const_node('cb_r', np.asarray([1, 1, 1, in_w], dtype=np.int64)),
        oh.make_node('Tile', ['one_col', 'cb_r'], ['tiled']),
        _const_node('cb_cs', np.asarray([0, 0],       dtype=np.int64)),
        _const_node('cb_ce', np.asarray([in_h, in_w], dtype=np.int64)),
        _const_node('cb_ca', np.asarray([2, 3],       dtype=np.int64)),
        oh.make_node('Slice', ['tiled', 'cb_cs', 'cb_ce', 'cb_ca'], ['cropped']),
    ]
    nodes += _pad_to_canvas_nodes('cropped', 'output', in_h, in_w, 'p')
    return _build_model_from_nodes(nodes, 'col_broadcast')

def make_cheap_border_add(grid_h, grid_w, border, fill_color):
    oh_ = grid_h + 2 * border; ow_ = grid_w + 2 * border
    if oh_ > H or ow_ > W: return None
    nodes  = _crop_nodes('input', 'content', grid_h, grid_w, 'pb0')
    nodes += [_const_node('pb_pads', np.asarray([0,0,border,border,0,0,border,border], dtype=np.int64)),
              oh.make_node('Pad', ['content', 'pb_pads'], ['padded'], mode='constant')]
    mask = np.zeros((1, C, oh_, ow_), dtype=np.float32)
    if 0 <= fill_color < C:
        mask[0, fill_color, :border, :] = 1.0; mask[0, fill_color, -border:, :] = 1.0
        mask[0, fill_color, :, :border] = 1.0; mask[0, fill_color, :, -border:] = 1.0
    nodes += [_const_node('pb_mask', mask), oh.make_node('Add', ['padded', 'pb_mask'], ['bordered'])]
    nodes += _pad_to_canvas_nodes('bordered', 'output', oh_, ow_, 'pbf')
    return _build_model_from_nodes(nodes, 'border_add')

def make_cheap_concat_rot180_w(in_h, in_w):
    nodes  = _crop_nodes('input', 'content', in_h, in_w, '0')
    nodes += _slice_reverse_nodes([2, 3], [in_h, in_w], 'content', 'rotated', 'r')
    nodes += [oh.make_node('Concat', ['content', 'rotated'], ['concated'], axis=3)]
    nodes += _pad_to_canvas_nodes('concated', 'output', in_h, in_w * 2, 'p')
    return _build_model_from_nodes(nodes, 'concat_rot180_w')

def make_cheap_crop_then_color(r0, c0, out_h, out_w, color_map):
    nodes = [
        _const_node('ctc_s', np.asarray([r0, c0], dtype=np.int64)),
        _const_node('ctc_e', np.asarray([r0 + out_h, c0 + out_w], dtype=np.int64)),
        _const_node('ctc_a', np.asarray([2, 3], dtype=np.int64)),
        oh.make_node('Slice', ['input', 'ctc_s', 'ctc_e', 'ctc_a'], ['cropped']),
    ]
    inv = {d: s for s, d in color_map.items()}
    idx  = np.arange(C, dtype=np.int64)
    mask = np.ones(C, dtype=np.float32)
    src_away = {s for s, d in color_map.items() if s != d}
    for d in range(C):
        if d in inv: idx[d] = inv[d]
        elif d in src_away: idx[d] = 0; mask[d] = 0.0
    nodes += [_const_node('gi_ctc', idx)]
    if mask.min() == 1.0:
        nodes += [oh.make_node('Gather', ['cropped', 'gi_ctc'], ['colored'], axis=1)]
    else:
        nodes += [oh.make_node('Gather', ['cropped', 'gi_ctc'], ['g_ctc'], axis=1),
                  _const_node('cm_ctc', mask.reshape(1, C, 1, 1)),
                  oh.make_node('Mul', ['g_ctc', 'cm_ctc'], ['colored'])]
    nodes += _pad_to_canvas_nodes('colored', 'output', out_h, out_w, 'pctc')
    return _build_model_from_nodes(nodes, 'crop_color')

SAFE_PAD_IDX = HW - 1

def _compact_mask(mask):
    if mask.ndim == 4 and mask.shape[0] == 1 and mask.shape[1] == C:
        if np.all(mask[0, 0:1] == mask[0, :]):
            return mask[:, 0:1].copy()
    return mask

def make_gather_onnx(gather_indices, mask=None):
    X = oh.make_tensor_value_info('input',  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info('output', TensorProto.FLOAT, [1, C, H, W])
    shape_chw      = onh.from_array(np.array([1, C, HW], dtype=np.int64), name='shape_chw')
    shape_chw_node = oh.make_node('Constant', [], ['shape_chw'], value=shape_chw)
    inp_reshape    = oh.make_node('Reshape', ['input', 'shape_chw'], ['inp_chw'])
    gi_tensor      = onh.from_array(gather_indices.astype(np.int64), name='gather_idx')
    gi_node        = oh.make_node('Constant', [], ['gather_idx'], value=gi_tensor)
    gather_node    = oh.make_node('Gather', ['inp_chw', 'gather_idx'], ['gathered'], axis=2)
    out_shape      = onh.from_array(np.array([1, C, H, W], dtype=np.int64), name='out_shape')
    out_shape_node = oh.make_node('Constant', [], ['out_shape'], value=out_shape)
    if mask is not None:
        reshape_back = oh.make_node('Reshape', ['gathered', 'out_shape'], ['gathered_4d'])
        mask = _compact_mask(mask)
        mask_tensor = onh.from_array(mask.astype(np.float32), name='mask')
        mask_node   = oh.make_node('Constant', [], ['mask'], value=mask_tensor)
        mul_node    = oh.make_node('Mul', ['gathered_4d', 'mask'], ['output'])
        nodes = [shape_chw_node, inp_reshape, gi_node, gather_node,
                 out_shape_node, reshape_back, mask_node, mul_node]
    else:
        reshape_back = oh.make_node('Reshape', ['gathered', 'out_shape'], ['output'])
        nodes = [shape_chw_node, inp_reshape, gi_node, gather_node, out_shape_node, reshape_back]
    graph = oh.make_graph(nodes, 'gather_net', [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid('', 17)])
    model.ir_version = 8
    return model

def make_const_onnx(const_output):
    X = oh.make_tensor_value_info('input',  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info('output', TensorProto.FLOAT, [1, C, H, W])
    const_tensor = onh.from_array(const_output.astype(np.float32), name='const_out')
    const_node   = oh.make_node('Constant', [], ['const_out'], value=const_tensor)
    id_node      = oh.make_node('Identity', ['const_out'], ['output'])
    graph = oh.make_graph([const_node, id_node], 'const_net', [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid('', 17)])
    model.ir_version = 8
    return model

def make_conv1x1_onnx(weight, bias=None):
    X = oh.make_tensor_value_info('input',  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info('output', TensorProto.FLOAT, [1, C, H, W])
    w_tensor = onh.from_array(weight.astype(np.float32), name='conv_weight')
    w_node   = oh.make_node('Constant', [], ['conv_weight'], value=w_tensor)
    if bias is not None:
        b_tensor  = onh.from_array(bias.astype(np.float32), name='conv_bias')
        b_node    = oh.make_node('Constant', [], ['conv_bias'], value=b_tensor)
        conv_node = oh.make_node('Conv', ['input', 'conv_weight', 'conv_bias'], ['output'],
                                 kernel_shape=[1, 1], pads=[0, 0, 0, 0])
        graph = oh.make_graph([w_node, b_node, conv_node], 'conv1x1', [X], [Y])
    else:
        conv_node = oh.make_node('Conv', ['input', 'conv_weight'], ['output'],
                                 kernel_shape=[1, 1], pads=[0, 0, 0, 0])
        graph = oh.make_graph([w_node, conv_node], 'conv1x1', [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid('', 17)])
    model.ir_version = 8
    return model

def make_channel_gather_onnx(gather_ch):
    X = oh.make_tensor_value_info('input',  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info('output', TensorProto.FLOAT, [1, C, H, W])
    gi_tensor = onh.from_array(gather_ch.astype(np.int32), name='gi')
    gi_node   = oh.make_node('Constant', [], ['gi'], value=gi_tensor)
    gather    = oh.make_node('Gather', ['input', 'gi'], ['output'], axis=1)
    graph = oh.make_graph([gi_node, gather], 'ch_gather', [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid('', 17)])
    model.ir_version = 8
    return model

def make_conv_onnx(weight, bias=None, kernel_size=3):
    pad = kernel_size // 2
    X = oh.make_tensor_value_info('input',  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info('output', TensorProto.FLOAT, [1, C, H, W])
    w_tensor = onh.from_array(weight.astype(np.float32), name='conv_weight')
    w_node   = oh.make_node('Constant', [], ['conv_weight'], value=w_tensor)
    if bias is not None:
        b_tensor  = onh.from_array(bias.astype(np.float32), name='conv_bias')
        b_node    = oh.make_node('Constant', [], ['conv_bias'], value=b_tensor)
        conv_node = oh.make_node('Conv', ['input', 'conv_weight', 'conv_bias'], ['output'],
                                 kernel_shape=[kernel_size, kernel_size], pads=[pad]*4)
        nodes = [w_node, b_node, conv_node]
    else:
        conv_node = oh.make_node('Conv', ['input', 'conv_weight'], ['output'],
                                 kernel_shape=[kernel_size, kernel_size], pads=[pad]*4)
        nodes = [w_node, conv_node]
    graph = oh.make_graph(nodes, 'conv_net', [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid('', 17)])
    model.ir_version = 8
    return model

def make_spatial_then_channel_gather_onnx(spatial_idx, channel_idx):
    X = oh.make_tensor_value_info('input',  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info('output', TensorProto.FLOAT, [1, C, H, W])
    shape_chw = onh.from_array(np.array([1, C, HW], dtype=np.int64), name='shape_chw')
    nodes = [
        oh.make_node('Constant', [], ['shape_chw'], value=shape_chw),
        oh.make_node('Reshape',  ['input', 'shape_chw'], ['inp_chw']),
        oh.make_node('Constant', [], ['si'], value=onh.from_array(spatial_idx.astype(np.int32), name='si')),
        oh.make_node('Gather',   ['inp_chw', 'si'], ['spatial_out'], axis=2),
        oh.make_node('Constant', [], ['out_shape'], value=onh.from_array(np.array([1, C, H, W], dtype=np.int64), name='out_shape')),
        oh.make_node('Reshape',  ['spatial_out', 'out_shape'], ['spatial_4d']),
        oh.make_node('Constant', [], ['ci'], value=onh.from_array(channel_idx.astype(np.int32), name='ci')),
        oh.make_node('Gather',   ['spatial_4d', 'ci'], ['output'], axis=1),
    ]
    graph = oh.make_graph(nodes, 'spatial_ch_gather', [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid('', 17)])
    model.ir_version = 8
    return model

def make_gather_then_conv1x1_onnx(gather_indices, conv_weight, mask=None):
    X = oh.make_tensor_value_info('input',  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info('output', TensorProto.FLOAT, [1, C, H, W])
    nodes = [
        oh.make_node('Constant', [], ['shape_chw'], value=onh.from_array(np.array([1, C, HW], dtype=np.int64), name='shape_chw')),
        oh.make_node('Reshape',  ['input', 'shape_chw'], ['inp_chw']),
        oh.make_node('Constant', [], ['gather_idx'], value=onh.from_array(gather_indices.astype(np.int64), name='gather_idx')),
        oh.make_node('Gather',   ['inp_chw', 'gather_idx'], ['gathered'], axis=2),
        oh.make_node('Constant', [], ['out_shape'], value=onh.from_array(np.array([1, C, H, W], dtype=np.int64), name='out_shape')),
        oh.make_node('Reshape',  ['gathered', 'out_shape'], ['gathered_4d']),
    ]
    w_tensor = onh.from_array(conv_weight.astype(np.float32), name='conv_weight')
    if mask is not None:
        mask = _compact_mask(mask)
        nodes += [
            oh.make_node('Constant', [], ['mask'], value=onh.from_array(mask.astype(np.float32), name='mask')),
            oh.make_node('Mul', ['gathered_4d', 'mask'], ['masked']),
            oh.make_node('Constant', [], ['conv_weight'], value=w_tensor),
            oh.make_node('Conv', ['masked', 'conv_weight'], ['output'], kernel_shape=[1,1], pads=[0,0,0,0]),
        ]
    else:
        nodes += [
            oh.make_node('Constant', [], ['conv_weight'], value=w_tensor),
            oh.make_node('Conv', ['gathered_4d', 'conv_weight'], ['output'], kernel_shape=[1,1], pads=[0,0,0,0]),
        ]
    graph = oh.make_graph(nodes, 'gather_conv1x1', [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid('', 17)])
    model.ir_version = 8
    return model

def make_symmetry_completion_onnx(gather_index_list, mask=None):
    X = oh.make_tensor_value_info('input',  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info('output', TensorProto.FLOAT, [1, C, H, W])
    ch_mask = np.ones((1, C, 1, 1), dtype=np.float32); ch_mask[0, 0, 0, 0] = 0.0
    nodes = [
        oh.make_node('Constant', [], ['ch_mask'], value=onh.from_array(ch_mask, name='ch_mask')),
        oh.make_node('Mul', ['input', 'ch_mask'], ['input_nobg']),
        oh.make_node('Constant', [], ['shape_chw'], value=onh.from_array(np.array([1, C, HW], dtype=np.int64), name='shape_chw')),
        oh.make_node('Reshape',  ['input_nobg', 'shape_chw'], ['inp_chw']),
        oh.make_node('Constant', [], ['out_shape'], value=onh.from_array(np.array([1, C, H, W], dtype=np.int64), name='out_shape')),
    ]
    reshape_names = []
    for i, gi in enumerate(gather_index_list):
        gn = f'gi_{i}'; rn = f'r_{i}'
        nodes += [
            oh.make_node('Constant', [], [gn], value=onh.from_array(gi.astype(np.int32), name=gn)),
            oh.make_node('Gather',   ['inp_chw', gn], [f'g_{i}'], axis=2),
            oh.make_node('Reshape',  [f'g_{i}', 'out_shape'], [rn]),
        ]
        reshape_names.append(rn)
    if len(reshape_names) == 1:
        nodes.append(oh.make_node('Identity', [reshape_names[0]], ['output']))
    else:
        prev = reshape_names[0]
        for i in range(1, len(reshape_names)):
            out = 'output' if (i == len(reshape_names) - 1 and mask is None) else f'max_{i}'
            nodes.append(oh.make_node('Max', [prev, reshape_names[i]], [out]))
            prev = out
    if mask is not None:
        mask = _compact_mask(mask)
        nodes += [
            oh.make_node('Constant', [], ['mask'], value=onh.from_array(mask.astype(np.float32), name='mask')),
            oh.make_node('Mul', [prev, 'mask'], ['output']),
        ]
    graph = oh.make_graph(nodes, 'sym_complete', [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid('', 17)])
    model.ir_version = 8
    return model

print('ONNX builders loaded.')

In [ ]:
def _slice_channel_node(in_name, out_name, ch_start, ch_end, suf=''):
    return [
        _const_node(f'sc_s{suf}', np.asarray([ch_start], dtype=np.int64)),
        _const_node(f'sc_e{suf}', np.asarray([ch_end],   dtype=np.int64)),
        _const_node(f'sc_a{suf}', np.asarray([1],        dtype=np.int64)),
        oh.make_node('Slice', [in_name, f'sc_s{suf}', f'sc_e{suf}', f'sc_a{suf}'], [out_name]),
    ]

def make_bbox_fill_onnx(fill_color):
    X = oh.make_tensor_value_info('input',  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info('output', TensorProto.FLOAT, [1, C, H, W])
    nodes  = _slice_channel_node('input', 'nonbg_slice', 1, C, 'nb')
    nodes += [oh.make_node('ReduceMax', ['nonbg_slice'], ['any_nonbg'], axes=[1], keepdims=1)]
    nodes += [oh.make_node('ReduceMax', ['any_nonbg'],   ['row_has'],   axes=[3], keepdims=1)]
    nodes += [oh.make_node('ReduceMax', ['any_nonbg'],   ['col_has'],   axes=[2], keepdims=1)]
    nodes += [oh.make_node('Mul', ['row_has', 'col_has'], ['bbox_mask'])]
    nodes += _slice_channel_node('input', 'bg_ch', 0, 1, 'bg')
    nodes += [oh.make_node('Mul', ['bbox_mask', 'bg_ch'], ['interior'])]
    w = np.zeros((C, 1, 1, 1), dtype=np.float32)
    w[0, 0, 0, 0] = -1.0; w[fill_color, 0, 0, 0] = 1.0
    nodes += [
        _const_node('bf_w', w),
        oh.make_node('Conv', ['interior', 'bf_w'], ['add_map'], kernel_shape=[1,1], pads=[0,0,0,0]),
        oh.make_node('Add',  ['input', 'add_map'], ['output']),
    ]
    graph = oh.make_graph(nodes, 'bbox_fill', [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid('', 17)])
    model.ir_version = 8
    return model

def make_hole_fill_onnx(fill_color, num_iters=14):
    X = oh.make_tensor_value_info('input',  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info('output', TensorProto.FLOAT, [1, C, H, W])
    nodes  = _slice_channel_node('input', 'bg_ch', 0, 1, 'hf1')
    border = np.zeros((1, 1, H, W), dtype=np.float32)
    border[0,0,0,:]=1.0; border[0,0,-1,:]=1.0; border[0,0,:,0]=1.0; border[0,0,:,-1]=1.0
    nodes += [_const_node('border_m', border),
              oh.make_node('Mul', ['bg_ch', 'border_m'], ['outside_0'])]
    prev = 'outside_0'
    for i in range(num_iters):
        nodes += [oh.make_node('MaxPool', [prev], [f'outside_mp_{i}'],
                               kernel_shape=[3,3], pads=[1,1,1,1], strides=[1,1]),
                  oh.make_node('Mul', [f'outside_mp_{i}', 'bg_ch'], [f'outside_{i+1}'])]
        prev = f'outside_{i+1}'
    nodes += [oh.make_node('Sub', ['bg_ch', prev], ['interior'])]
    w = np.zeros((C, 1, 1, 1), dtype=np.float32)
    w[0, 0, 0, 0] = -1.0; w[fill_color, 0, 0, 0] = 1.0
    nodes += [_const_node('hf_w', w),
              oh.make_node('Conv', ['interior', 'hf_w'], ['add_map'], kernel_shape=[1,1], pads=[0,0,0,0]),
              oh.make_node('Add',  ['input', 'add_map'], ['output'])]
    graph = oh.make_graph(nodes, 'hole_fill', [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid('', 17)])
    model.ir_version = 8
    return model

def make_outline_onnx():
    X = oh.make_tensor_value_info('input',  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info('output', TensorProto.FLOAT, [1, C, H, W])
    nodes  = _slice_channel_node('input', 'nonbg_slice', 1, C, 'o2')
    nodes += [
        oh.make_node('ReduceMax', ['nonbg_slice'], ['any_nonbg'], axes=[1], keepdims=1),
        _const_node('ol_one_s', np.asarray([[[[1.0]]]], dtype=np.float32)),
        oh.make_node('Sub', ['ol_one_s', 'any_nonbg'], ['neg_anb']),
        _const_node('ol_pads',  np.asarray([0,0,1,1,0,0,1,1], dtype=np.int64)),
        _const_node('ol_pad_v', np.asarray(1.0, dtype=np.float32)),
        oh.make_node('Pad', ['neg_anb', 'ol_pads', 'ol_pad_v'], ['neg_padded'], mode='constant'),
        oh.make_node('MaxPool', ['neg_padded'], ['has_empty_near'],
                     kernel_shape=[3,3], pads=[0,0,0,0], strides=[1,1]),
        oh.make_node('Mul', ['any_nonbg', 'has_empty_near'], ['boundary']),
        oh.make_node('Sub', ['any_nonbg', 'boundary'], ['interior_nonbg']),
        oh.make_node('Sub', ['ol_one_s',  'interior_nonbg'], ['keep_mask']),
        oh.make_node('Mul', ['input', 'keep_mask'], ['stripped']),
    ]
    w = np.zeros((C, 1, 1, 1), dtype=np.float32); w[0, 0, 0, 0] = 1.0
    nodes += [_const_node('ol_w', w),
              oh.make_node('Conv', ['interior_nonbg', 'ol_w'], ['bg_add'], kernel_shape=[1,1], pads=[0,0,0,0]),
              oh.make_node('Add',  ['stripped', 'bg_add'], ['output'])]
    graph = oh.make_graph(nodes, 'outline', [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid('', 17)])
    model.ir_version = 8
    return model

def make_keep_largest_color_onnx():
    X = oh.make_tensor_value_info('input',  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info('output', TensorProto.FLOAT, [1, C, H, W])
    nodes = [
        _const_node('klc_axes', np.asarray([2, 3], dtype=np.int64)),
        oh.make_node('ReduceSum', ['input', 'klc_axes'], ['areas'], keepdims=1),
    ]
    ch_mask = np.ones((1, C, 1, 1), dtype=np.float32); ch_mask[0, 0, 0, 0] = 0.0
    nodes += [_const_node('ch_m', ch_mask),
              oh.make_node('Mul', ['areas', 'ch_m'], ['areas_nbg']),
              oh.make_node('ReduceMax', ['areas_nbg'], ['max_area'], axes=[1], keepdims=1),
              oh.make_node('Equal', ['areas_nbg', 'max_area'], ['is_max']),
              oh.make_node('Cast',  ['is_max'], ['is_max_f'], to=TensorProto.FLOAT),
              oh.make_node('Mul',   ['input', 'is_max_f'], ['masked_in'])]
    nodes += _slice_channel_node('input',     'nonbg_in',  1, C, 'klc')
    nodes += [oh.make_node('ReduceMax', ['nonbg_in'],  ['any_nb_in'],  axes=[1], keepdims=1)]
    nodes += _slice_channel_node('masked_in', 'nonbg_out', 1, C, 'klc2')
    nodes += [oh.make_node('ReduceMax', ['nonbg_out'], ['any_nb_out'], axes=[1], keepdims=1),
              oh.make_node('Sub', ['any_nb_in', 'any_nb_out'], ['removed'])]
    w = np.zeros((C, 1, 1, 1), dtype=np.float32); w[0, 0, 0, 0] = 1.0
    nodes += [_const_node('klc_w', w),
              oh.make_node('Conv', ['removed', 'klc_w'], ['bg_add'], kernel_shape=[1,1], pads=[0,0,0,0]),
              oh.make_node('Add',  ['masked_in', 'bg_add'], ['output'])]
    graph = oh.make_graph(nodes, 'keep_largest', [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid('', 17)])
    model.ir_version = 8
    return model

def make_keep_smallest_color_onnx():
    X = oh.make_tensor_value_info('input',  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info('output', TensorProto.FLOAT, [1, C, H, W])
    nodes = [
        _const_node('ksc_axes', np.asarray([2, 3], dtype=np.int64)),
        oh.make_node('ReduceSum', ['input', 'ksc_axes'], ['areas'], keepdims=1),
    ]
    ch_mask = np.zeros((1, C, 1, 1), dtype=np.float32); ch_mask[0, 0, 0, 0] = 1e9
    nodes += [_const_node('big_off', ch_mask)]
    zero_t = np.zeros((1, 1, 1, 1), dtype=np.float32)
    nodes += [
        _const_node('zero_k', zero_t),
        oh.make_node('Equal', ['areas', 'zero_k'], ['is_zero']),
        oh.make_node('Cast',  ['is_zero'], ['is_zero_f'], to=TensorProto.FLOAT),
        oh.make_node('Add',   ['areas', 'big_off'], ['areas_bg_big']),
        _const_node('big_k', np.full((1,1,1,1), 1e9, dtype=np.float32)),
        oh.make_node('Mul', ['is_zero_f', 'big_k'], ['big_for_zero']),
        oh.make_node('Add', ['areas_bg_big', 'big_for_zero'], ['areas_adj']),
        oh.make_node('ReduceMin', ['areas_adj'], ['min_area'], axes=[1], keepdims=1),
        oh.make_node('Equal', ['areas_adj', 'min_area'], ['is_min']),
        oh.make_node('Cast',  ['is_min'], ['is_min_f'], to=TensorProto.FLOAT),
        oh.make_node('Mul',   ['input', 'is_min_f'], ['masked_in']),
    ]
    nodes += _slice_channel_node('input',     'nb_in',  1, C, 'ks')
    nodes += [oh.make_node('ReduceMax', ['nb_in'],  ['any_nb_in'],  axes=[1], keepdims=1)]
    nodes += _slice_channel_node('masked_in', 'nb_out', 1, C, 'ks2')
    nodes += [oh.make_node('ReduceMax', ['nb_out'], ['any_nb_out'], axes=[1], keepdims=1),
              oh.make_node('Sub', ['any_nb_in', 'any_nb_out'], ['removed'])]
    w = np.zeros((C, 1, 1, 1), dtype=np.float32); w[0, 0, 0, 0] = 1.0
    nodes += [_const_node('ks_w', w),
              oh.make_node('Conv', ['removed', 'ks_w'], ['bg_add'], kernel_shape=[1,1], pads=[0,0,0,0]),
              oh.make_node('Add',  ['masked_in', 'bg_add'], ['output'])]
    graph = oh.make_graph(nodes, 'keep_smallest', [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid('', 17)])
    model.ir_version = 8
    return model

print('V11 primitives loaded.')

In [ ]:
def make_period_row_tile_onnx(period, in_h, in_w, out_h):
    """Slice first `period` rows, tile vertically to out_h."""
    nodes = [
        _const_node('prt_s', np.asarray([0, 0],       dtype=np.int64)),
        _const_node('prt_e', np.asarray([period, in_w], dtype=np.int64)),
        _const_node('prt_a', np.asarray([2, 3],       dtype=np.int64)),
        oh.make_node('Slice', ['input', 'prt_s', 'prt_e', 'prt_a'], ['base']),
        _const_node('prt_r', np.asarray([1, 1, out_h // period, 1], dtype=np.int64)),
        oh.make_node('Tile', ['base', 'prt_r'], ['tiled']),
    ]
    nodes += _pad_to_canvas_nodes('tiled', 'output', out_h, in_w, 'p')
    return _build_model_from_nodes(nodes, 'period_row_tile')

def make_period_col_tile_onnx(period, in_h, in_w, out_w):
    """Slice first `period` cols, tile horizontally to out_w."""
    nodes = [
        _const_node('pct_s', np.asarray([0, 0],       dtype=np.int64)),
        _const_node('pct_e', np.asarray([in_h, period], dtype=np.int64)),
        _const_node('pct_a', np.asarray([2, 3],       dtype=np.int64)),
        oh.make_node('Slice', ['input', 'pct_s', 'pct_e', 'pct_a'], ['base']),
        _const_node('pct_r', np.asarray([1, 1, 1, out_w // period], dtype=np.int64)),
        oh.make_node('Tile', ['base', 'pct_r'], ['tiled']),
    ]
    nodes += _pad_to_canvas_nodes('tiled', 'output', in_h, out_w, 'p')
    return _build_model_from_nodes(nodes, 'period_col_tile')

def make_bbox_tile_onnx(r0, c0, crop_h, crop_w, tr, tc):
    """Crop bbox then tile tr x tc."""
    nodes = [
        _const_node('bbt_s', np.asarray([r0, c0],           dtype=np.int64)),
        _const_node('bbt_e', np.asarray([r0+crop_h, c0+crop_w], dtype=np.int64)),
        _const_node('bbt_a', np.asarray([2, 3],             dtype=np.int64)),
        oh.make_node('Slice', ['input', 'bbt_s', 'bbt_e', 'bbt_a'], ['cropped']),
        _const_node('bbt_r', np.asarray([1, 1, tr, tc], dtype=np.int64)),
        oh.make_node('Tile', ['cropped', 'bbt_r'], ['tiled']),
    ]
    nodes += _pad_to_canvas_nodes('tiled', 'output', crop_h*tr, crop_w*tc, 'p')
    return _build_model_from_nodes(nodes, 'bbox_tile')

def make_crop_scale_onnx(r0, c0, crop_h, crop_w, factor):
    """Crop fixed region then nearest-neighbor upscale."""
    nodes = [
        _const_node('cs2_s', np.asarray([r0, c0],           dtype=np.int64)),
        _const_node('cs2_e', np.asarray([r0+crop_h, c0+crop_w], dtype=np.int64)),
        _const_node('cs2_a', np.asarray([2, 3],             dtype=np.int64)),
        oh.make_node('Slice', ['input', 'cs2_s', 'cs2_e', 'cs2_a'], ['cropped']),
        _const_node('cs2_roi',    np.asarray([], dtype=np.float32)),
        _const_node('cs2_scales', np.asarray([], dtype=np.float32)),
        _const_node('cs2_sizes',  np.asarray([1, C, crop_h*factor, crop_w*factor], dtype=np.int64)),
        oh.make_node('Resize', ['cropped','cs2_roi','cs2_scales','cs2_sizes'], ['scaled'],
                     mode='nearest', coordinate_transformation_mode='asymmetric',
                     nearest_mode='floor'),
    ]
    nodes += _pad_to_canvas_nodes('scaled', 'output', crop_h*factor, crop_w*factor, 'p')
    return _build_model_from_nodes(nodes, 'crop_scale')

def make_frame_border_onnx(border_color, grid_h, grid_w):
    """Draw 1-pixel border of border_color over the input."""
    keep = np.ones((1, 1, H, W), dtype=np.float32)
    keep[0,0,0,:grid_w]     = 0.0; keep[0,0,grid_h-1,:grid_w] = 0.0
    keep[0,0,:grid_h,0]     = 0.0; keep[0,0,:grid_h,grid_w-1] = 0.0
    mask = np.zeros((1, C, H, W), dtype=np.float32)
    if 0 <= border_color < C:
        mask[0,border_color,0,:grid_w]          = 1.0
        mask[0,border_color,grid_h-1,:grid_w]   = 1.0
        mask[0,border_color,:grid_h,0]          = 1.0
        mask[0,border_color,:grid_h,grid_w-1]   = 1.0
    X = oh.make_tensor_value_info('input',  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info('output', TensorProto.FLOAT, [1, C, H, W])
    kt = onh.from_array(keep, name='keep'); kn = oh.make_node('Constant', [], ['keep'], value=kt)
    mt = onh.from_array(mask, name='brd');  mn = oh.make_node('Constant', [], ['brd'],  value=mt)
    mul = oh.make_node('Mul', ['input', 'keep'], ['interior'])
    add = oh.make_node('Add', ['interior', 'brd'], ['output'])
    graph = oh.make_graph([kn, mn, mul, add], 'frame_border', [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid('', 17)])
    model.ir_version = 8
    return model

def make_row_select_tile_onnx(row_idx, in_w, out_h):
    """Extract row_idx, optionally tile to out_h rows."""
    nodes = [
        _const_node('rst_s', np.asarray([row_idx],   dtype=np.int64)),
        _const_node('rst_e', np.asarray([row_idx+1], dtype=np.int64)),
        _const_node('rst_a', np.asarray([2],         dtype=np.int64)),
        oh.make_node('Slice', ['input', 'rst_s', 'rst_e', 'rst_a'], ['row']),
    ]
    if out_h > 1:
        nodes += [
            _const_node('rst_r', np.asarray([1, 1, out_h, 1], dtype=np.int64)),
            oh.make_node('Tile', ['row', 'rst_r'], ['tiled']),
        ]
        nodes += _pad_to_canvas_nodes('tiled', 'output', out_h, in_w, 'p')
    else:
        nodes += _pad_to_canvas_nodes('row', 'output', 1, in_w, 'p')
    return _build_model_from_nodes(nodes, 'row_select_tile')

print('New ONNX builders (CHANGE 4) loaded.')

In [ ]:
def detect_bbox_fill(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    fill = None
    for p in pairs:
        ig = np.array(p['input']); og = np.array(p['output'])
        mask = ig != 0
        if not mask.any():
            if not np.array_equal(ig, og): return None
            continue
        rs, cs = np.where(mask)
        r0, r1 = rs.min(), rs.max(); c0, c1 = cs.min(), cs.max()
        inside = np.zeros_like(ig, dtype=bool)
        inside[r0:r1+1, c0:c1+1] = True
        if not np.array_equal(ig[~inside], og[~inside]): return None
        changed = (ig != og) & inside
        if np.any(ig[changed] != 0): return None
        nonbg_inside = (ig != 0) & inside
        if not np.array_equal(ig[nonbg_inside], og[nonbg_inside]): return None
        new_vals = og[changed]
        if new_vals.size == 0: continue
        u = np.unique(new_vals)
        if len(u) != 1: return None
        if fill is None: fill = int(u[0])
        elif fill != int(u[0]): return None
    return fill

def detect_hole_fill(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    from collections import deque
    fill = None
    for p in pairs:
        ig = np.array(p['input']); og = np.array(p['output'])
        h, w = ig.shape
        outside = np.zeros_like(ig, dtype=bool)
        dq = deque()
        for c in range(w):
            if ig[0,c]==0:   dq.append((0,c));   outside[0,c]=True
            if ig[h-1,c]==0: dq.append((h-1,c)); outside[h-1,c]=True
        for r in range(h):
            if ig[r,0]==0:   dq.append((r,0));   outside[r,0]=True
            if ig[r,w-1]==0: dq.append((r,w-1)); outside[r,w-1]=True
        while dq:
            r, c = dq.popleft()
            for dr, dc in [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]:
                rr, cc = r+dr, c+dc
                if 0<=rr<h and 0<=cc<w and not outside[rr,cc] and ig[rr,cc]==0:
                    outside[rr,cc]=True; dq.append((rr,cc))
        interior = (ig == 0) & ~outside
        if not np.array_equal(ig[~interior], og[~interior]): return None
        if not interior.any(): continue
        u = np.unique(og[interior])
        if len(u) != 1: return None
        if fill is None: fill = int(u[0])
        elif fill != int(u[0]): return None
    return fill

def detect_outline_extract(pairs):
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig = np.array(p['input']); og = np.array(p['output'])
        h, w = ig.shape; exp = ig.copy()
        for r in range(h):
            for c in range(w):
                if ig[r,c] == 0: continue
                if r==0 or r==h-1 or c==0 or c==w-1: continue
                if not any(ig[r+dr,c+dc]==0 for dr,dc in [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]
                           if 0<=r+dr<h and 0<=c+dc<w):
                    exp[r,c] = 0
        if not np.array_equal(exp, og): return False
    return True

def detect_keep_largest_color(pairs):
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig = np.array(p['input']); og = np.array(p['output'])
        counts = {c: int(np.sum(ig==c)) for c in range(1, C)}
        if all(v==0 for v in counts.values()): return False
        max_c = max(counts, key=counts.get)
        if not np.array_equal(np.where(ig==max_c, ig, 0), og): return False
    return True

def detect_keep_smallest_color(pairs):
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig = np.array(p['input']); og = np.array(p['output'])
        counts = {c: int(np.sum(ig==c)) for c in range(1, C) if np.sum(ig==c) > 0}
        if not counts: return False
        min_c = min(counts, key=counts.get)
        if not np.array_equal(np.where(ig==min_c, ig, 0), og): return False
    return True

# ---- NEW detectors (CHANGE 4) ----

def detect_period_row_tiling(pairs):
    """Input rows have period P; output tiles them to larger height."""
    for p in pairs:
        ig = np.array(p['input']); og = np.array(p['output'])
        ih, iw = ig.shape; oh_, ow_ = og.shape
        if ow_ != iw: return None
        for period in range(1, ih + 1):
            if ih % period != 0 or oh_ % period != 0: continue
            base = ig[:period, :]
            if not np.array_equal(np.tile(base, (ih // period, 1)), ig): continue
            if np.array_equal(np.tile(base, (oh_ // period, 1)), og):
                return (period, ih, iw, oh_)
    return None

def detect_period_col_tiling(pairs):
    """Input cols have period P; output tiles them to larger width."""
    for p in pairs:
        ig = np.array(p['input']); og = np.array(p['output'])
        ih, iw = ig.shape; oh_, ow_ = og.shape
        if ih != oh_: return None
        for period in range(1, iw + 1):
            if iw % period != 0 or ow_ % period != 0: continue
            base = ig[:, :period]
            if not np.array_equal(np.tile(base, (1, iw // period)), ig): continue
            if np.array_equal(np.tile(base, (1, ow_ // period)), og):
                return (period, ih, iw, ow_)
    return None

def detect_bbox_then_tile(pairs):
    """Output = tight bbox crop, then tiled to fill output."""
    results = []
    for p in pairs:
        ig = np.array(p['input']); og = np.array(p['output'])
        mask = ig != 0
        if not mask.any(): return None
        rs, cs = np.where(mask)
        r0, r1, c0, c1 = int(rs.min()), int(rs.max()), int(cs.min()), int(cs.max())
        crop_h, crop_w = r1 - r0 + 1, c1 - c0 + 1
        crop = ig[r0:r1+1, c0:c1+1]
        oh_, ow_ = og.shape
        found = False
        for tr in range(1, oh_ // max(crop_h, 1) + 1):
            for tc in range(1, ow_ // max(crop_w, 1) + 1):
                if crop_h * tr == oh_ and crop_w * tc == ow_:
                    if np.array_equal(np.tile(crop, (tr, tc)), og):
                        results.append((r0, c0, crop_h, crop_w, tr, tc))
                        found = True; break
            if found: break
        if not found: return None
    if not results: return None
    if len(set(results)) == 1: return results[0]
    return None

def detect_crop_then_scale(pairs):
    """Output = fixed-position crop then nearest-neighbor upscale."""
    if not pairs: return None
    oh_, ow_ = np.array(pairs[0]['output']).shape
    for factor in [2, 3, 4, 5]:
        ch, cw = oh_ // factor, ow_ // factor
        if ch < 1 or cw < 1: continue
        if ch * factor != oh_ or cw * factor != ow_: continue
        ih, iw = np.array(pairs[0]['input']).shape
        for r0 in range(ih - ch + 1):
            for c0 in range(iw - cw + 1):
                if all(
                    np.array_equal(
                        np.repeat(np.repeat(np.array(p['input'])[r0:r0+ch, c0:c0+cw], factor, 0), factor, 1),
                        np.array(p['output'])
                    ) for p in pairs
                ):
                    return (r0, c0, ch, cw, factor)
    return None

def detect_frame_border(pairs):
    """Output = input with 1-pixel border of fixed color."""
    if not all(shapes_match(p) for p in pairs): return None
    color = None
    for p in pairs:
        ig = np.array(p['input']); og = np.array(p['output'])
        if not np.array_equal(ig[1:-1, 1:-1], og[1:-1, 1:-1]): return None
        edge_vals = (set(og[0,:].tolist()) | set(og[-1,:].tolist()) |
                     set(og[:,0].tolist())  | set(og[:,-1].tolist()))
        if len(edge_vals) != 1: return None
        c_ = int(edge_vals.pop())
        if color is None: color = c_
        elif color != c_: return None
    return color

def detect_fixed_row_select(pairs):
    """Output = one specific row of input, optionally tiled."""
    if not pairs: return None
    results = []
    for p in pairs:
        ig = np.array(p['input']); og = np.array(p['output'])
        ih, iw = ig.shape; oh_, ow_ = og.shape
        if ow_ != iw: return None
        found = None
        for r in range(ih):
            if oh_ == 1 and np.array_equal(ig[r:r+1, :], og): found = r; break
            if oh_ > 1  and np.array_equal(np.tile(ig[r:r+1, :], (oh_, 1)), og): found = r; break
        if found is None: return None
        results.append((found, oh_))
    if len(set(results)) == 1: return results[0]
    return None

# ---- Original detectors (unchanged) ----

def _connected_components_nonbg(g, bg=0):
    h, w = g.shape; visited = np.zeros_like(g, dtype=bool); comps = []
    for r in range(h):
        for c in range(w):
            if visited[r,c] or g[r,c]==bg: continue
            stack = [(r,c)]; cells = []
            while stack:
                rr, cc = stack.pop()
                if not (0<=rr<h and 0<=cc<w): continue
                if visited[rr,cc] or g[rr,cc]==bg: continue
                visited[rr,cc]=True; cells.append((rr,cc))
                stack.extend([(rr+1,cc),(rr-1,cc),(rr,cc+1),(rr,cc-1)])
            comps.append(cells)
    return comps

def detect_rotation(pairs):
    for angle in [90, 180, 270]:
        k = angle // 90
        if all(np.rot90(np.array(p['input']),k).shape==np.array(p['output']).shape and
               np.array_equal(np.rot90(np.array(p['input']),k),np.array(p['output'])) for p in pairs):
            return angle
    return None

def detect_flip(pairs):
    for axis_val, name in [(0,'vertical'),(1,'horizontal'),(-1,'both')]:
        def do_flip(ig, a=axis_val):
            return np.flip(np.flip(ig,0),1) if a==-1 else np.flip(ig,axis=a)
        if all(do_flip(np.array(p['input'])).shape==np.array(p['output']).shape and
               np.array_equal(do_flip(np.array(p['input'])),np.array(p['output'])) for p in pairs):
            return name
    return None

def detect_transpose(pairs):
    return all(np.array(p['input']).T.shape==np.array(p['output']).shape and
               np.array_equal(np.array(p['input']).T,np.array(p['output'])) for p in pairs)

def detect_anti_transpose(pairs):
    return all(np.flip(np.flip(np.array(p['input']).T,0),1).shape==np.array(p['output']).shape and
               np.array_equal(np.flip(np.flip(np.array(p['input']).T,0),1),np.array(p['output'])) for p in pairs)

def detect_crop(pairs):
    results = []
    for p in pairs:
        ig=np.array(p['input']); og=np.array(p['output'])
        ih,iw=ig.shape; oh_,ow_=og.shape
        if oh_>ih or ow_>iw: return None
        found=False
        for r in range(ih-oh_+1):
            for c in range(iw-ow_+1):
                if np.array_equal(ig[r:r+oh_,c:c+ow_],og):
                    results.append((r,c,oh_,ow_)); found=True; break
            if found: break
        if not found: return None
    if len(set((r[0],r[1]) for r in results))==1: return results[0]
    return None

def detect_tile(pairs):
    for p in pairs:
        ig=np.array(p['input']); og=np.array(p['output'])
        ih,iw=ig.shape; oh_,ow_=og.shape
        if oh_%ih!=0 or ow_%iw!=0: return None
        tr=oh_//ih; tc=ow_//iw
        if not np.array_equal(np.tile(ig,(tr,tc)),og): return None
    return (np.array(pairs[0]['output']).shape[0]//np.array(pairs[0]['input']).shape[0],
            np.array(pairs[0]['output']).shape[1]//np.array(pairs[0]['input']).shape[1])

def detect_scale(pairs):
    for factor in [2,3,4,5]:
        if all(np.array(p['output']).shape==(np.array(p['input']).shape[0]*factor,np.array(p['input']).shape[1]*factor) and
               np.array_equal(np.repeat(np.repeat(np.array(p['input']),factor,0),factor,1),np.array(p['output'])) for p in pairs):
            return factor
    return None

def detect_color_replace(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    replacements = {}
    for p in pairs:
        for a,b in zip(np.array(p['input']).flat, np.array(p['output']).flat):
            if a!=b:
                if a in replacements and replacements[a]!=b: return None
                replacements[a]=b
    if not replacements: return None
    for p in pairs:
        ig=np.array(p['input']); exp=ig.copy()
        for s,d in replacements.items(): exp[ig==s]=d
        if not np.array_equal(exp,np.array(p['output'])): return None
    return replacements

def detect_color_swap(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    swap_pairs=set()
    for p in pairs:
        for a,b in zip(np.array(p['input']).flat,np.array(p['output']).flat):
            if a!=b: swap_pairs.add((min(a,b),max(a,b)))
    if len(swap_pairs)!=1: return None
    a,b=list(swap_pairs)[0]; saw_ab=saw_ba=False
    for p in pairs:
        ig=np.array(p['input']); og=np.array(p['output'])
        exp=ig.copy(); exp[ig==a]=b; exp[ig==b]=a
        if not np.array_equal(exp,og): return None
        if np.any((ig==a)&(og==b)): saw_ab=True
        if np.any((ig==b)&(og==a)): saw_ba=True
    return (a,b) if saw_ab and saw_ba else None

def detect_pixel_permutation(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    if len(set(np.array(p['input']).shape for p in pairs))!=1: return None
    ih,iw=np.array(pairs[0]['input']).shape
    if ih>H or iw>W: return None
    all_inputs  = [np.array(p['input']).flatten()  for p in pairs]
    all_outputs = [np.array(p['output']).flatten() for p in pairs]
    grid_size = ih*iw; assignment={}
    for dst_idx in range(grid_size):
        targets=[all_outputs[pi][dst_idx] for pi in range(len(pairs))]
        candidates=None
        for pi in range(len(pairs)):
            match=set(np.where(all_inputs[pi]==targets[pi])[0].tolist())
            candidates=match if candidates is None else candidates&match
            if not candidates: return None
        if len(candidates)!=1: return None
        assignment[dst_idx]=candidates.pop()
    gi=np.arange(HW,dtype=np.int64)
    for dst,src in assignment.items():
        dr,dc=divmod(dst,iw); sr,sc=divmod(src,iw)
        gi[dr*W+dc]=sr*W+sc
    for p in pairs:
        ig=np.array(p['input']); padded=np.zeros((H,W),dtype=np.int32)
        padded[:ih,:iw]=ig
        pred=padded.flatten()[gi].reshape(H,W)[:ih,:iw]
        if not np.array_equal(pred,np.array(p['output'])): return None
    return gi

def detect_mirror_h_concat(pairs):
    for p in pairs:
        ig=np.array(p['input']); og=np.array(p['output'])
        if og.shape!=(ig.shape[0],ig.shape[1]*2): return False
        if not np.array_equal(np.concatenate([ig,np.flip(ig,1)],1),og): return False
    return True

def detect_mirror_v_concat(pairs):
    for p in pairs:
        ig=np.array(p['input']); og=np.array(p['output'])
        if og.shape!=(ig.shape[0]*2,ig.shape[1]): return False
        if not np.array_equal(np.concatenate([ig,np.flip(ig,0)],0),og): return False
    return True

def detect_self_concat_h(pairs):
    for p in pairs:
        ig=np.array(p['input']); og=np.array(p['output'])
        if og.shape!=(ig.shape[0],ig.shape[1]*2): return False
        if not np.array_equal(np.concatenate([ig,ig],1),og): return False
    return True

def detect_self_concat_v(pairs):
    for p in pairs:
        ig=np.array(p['input']); og=np.array(p['output'])
        if og.shape!=(ig.shape[0]*2,ig.shape[1]): return False
        if not np.array_equal(np.concatenate([ig,ig],0),og): return False
    return True

def detect_quad_mirror(pairs):
    for p in pairs:
        ig=np.array(p['input']); og=np.array(p['output']); ih,iw=ig.shape
        if og.shape!=(ih*2,iw*2): return None
        exp=np.block([[ig,np.flip(ig,1)],[np.flip(ig,0),np.flip(np.flip(ig,0),1)]])
        if not np.array_equal(exp,og): return None
    return 'tl'

def detect_h_symmetry_complete(pairs):
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig=np.array(p['input']); og=np.array(p['output'])
        if not np.array_equal(np.where(ig!=0,ig,np.flip(ig,1)),og): return False
    return True

def detect_v_symmetry_complete(pairs):
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig=np.array(p['input']); og=np.array(p['output'])
        if not np.array_equal(np.where(ig!=0,ig,np.flip(ig,0)),og): return False
    return True

def detect_both_symmetry_complete(pairs):
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig=np.array(p['input']); og=np.array(p['output'])
        stack=np.stack([ig,np.flip(ig,1),np.flip(ig,0),np.flip(np.flip(ig,0),1)],0)
        if not np.array_equal(np.where(ig!=0,ig,stack.max(0)),og): return False
    return True

def detect_diag_symmetry_complete(pairs):
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig=np.array(p['input']); og=np.array(p['output'])
        if ig.shape[0]!=ig.shape[1]: return False
        if not np.array_equal(np.where(ig!=0,ig,ig.T),og): return False
    return True

def detect_translation(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    ih,iw=np.array(pairs[0]['input']).shape
    for dr in range(-min(ih,11)+1,min(ih,11)):
        for dc in range(-min(iw,11)+1,min(iw,11)):
            if dr==0 and dc==0: continue
            ok=True
            for p in pairs:
                ig=np.array(p['input']); og=np.array(p['output'])
                hh,ww=ig.shape; s=np.zeros_like(ig)
                sr0=max(0,-dr);sr1=min(hh,hh-dr);sc0=max(0,-dc);sc1=min(ww,ww-dc)
                dr0=max(0,dr);dc0=max(0,dc);rh=sr1-sr0;rw=sc1-sc0
                if rh>0 and rw>0: s[dr0:dr0+rh,dc0:dc0+rw]=ig[sr0:sr1,sc0:sc1]
                if not np.array_equal(s,og): ok=False; break
            if ok: return (dr,dc)
    return None

def detect_downscale(pairs):
    for f in [2,3,4,5]:
        if all(np.array(p['input']).shape==(np.array(p['output']).shape[0]*f,np.array(p['output']).shape[1]*f) and
               np.array_equal(np.array(p['input'])[::f,::f],np.array(p['output'])) for p in pairs):
            return f
    return None

def detect_color_filter(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    cand=None
    for p in pairs:
        ig=np.array(p['input']); og=np.array(p['output'])
        diff=(ig!=og)
        if not np.any(diff): continue
        if not np.all(og[diff]==0): return None
        local=set(int(x) for x in ig[diff])
        cand=local if cand is None else cand|local
    if not cand: return None
    for p in pairs:
        ig=np.array(p['input']).copy()
        for c in cand: ig[ig==c]=0
        if not np.array_equal(ig,np.array(p['output'])): return None
    return cand

def detect_row_broadcast(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    chosen=None
    for p in pairs:
        ig=np.array(p['input']); og=np.array(p['output']); m=None
        for r in range(ig.shape[0]):
            if np.all(og==ig[r,:]): m=r; break
        if m is None: return None
        if chosen is None: chosen=m
        elif chosen!=m: return None
    return chosen

def detect_col_broadcast(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    chosen=None
    for p in pairs:
        ig=np.array(p['input']); og=np.array(p['output']); m=None
        for c in range(ig.shape[1]):
            if np.all(og==ig[:,c:c+1]): m=c; break
        if m is None: return None
        if chosen is None: chosen=m
        elif chosen!=m: return None
    return chosen

def detect_border_add(pairs):
    if not pairs: return None
    fi=np.array(pairs[0]['input']); fo=np.array(pairs[0]['output'])
    ih,iw=fi.shape; oh_,ow_=fo.shape
    if oh_-ih!=ow_-iw or oh_<=ih: return None
    d=oh_-ih
    if d%2: return None
    b=d//2
    if not np.array_equal(fo[b:b+ih,b:b+iw],fi): return None
    vals=set(fo[:b,:].flat)|set(fo[-b:,:].flat)|set(fo[:,:b].flat)|set(fo[:,-b:].flat)
    if len(vals)!=1: return None
    fill=int(vals.pop())
    for p in pairs[1:]:
        ig=np.array(p['input']); og=np.array(p['output'])
        if og.shape!=(ig.shape[0]+2*b,ig.shape[1]+2*b): return None
        if not np.array_equal(og[b:b+ig.shape[0],b:b+ig.shape[1]],ig): return None
        m2=np.zeros(og.shape,dtype=bool);m2[:b,:]=1;m2[-b:,:]=1;m2[:,:b]=1;m2[:,-b:]=1
        if not np.all(og[m2]==fill): return None
    return (b,fill)

def detect_crop_plus_color(pairs):
    if not pairs: return None
    fi=np.array(pairs[0]['input']); fo=np.array(pairs[0]['output'])
    oh_,ow_=fo.shape; ih,iw=fi.shape
    if oh_>ih or ow_>iw: return None
    for r0 in range(ih-oh_+1):
        for c0 in range(iw-ow_+1):
            cm={}; ok=True
            for p in pairs:
                ig=np.array(p['input']); og=np.array(p['output'])
                if og.shape!=(oh_,ow_) or r0+oh_>ig.shape[0] or c0+ow_>ig.shape[1]: ok=False; break
                for a,b in zip(ig[r0:r0+oh_,c0:c0+ow_].flat,og.flat):
                    a,b=int(a),int(b)
                    if a in cm and cm[a]!=b: ok=False; break
                    cm[a]=b
                if not ok: break
            if not ok: continue
            if all(k==v for k,v in cm.items()): continue
            v=True
            for p in pairs:
                ig=np.array(p['input']); og=np.array(p['output'])
                reg=ig[r0:r0+oh_,c0:c0+ow_].copy()
                for k2,v2 in cm.items(): reg[reg==k2]=v2
                if not np.array_equal(reg,og): v=False; break
            if v: return (r0,c0,oh_,ow_,cm)
    return None

def detect_concat_rot180_w(pairs):
    for p in pairs:
        ig=np.array(p['input']); og=np.array(p['output'])
        if og.shape!=(ig.shape[0],ig.shape[1]*2): return None
        if not np.array_equal(np.concatenate([ig,np.rot90(ig,2)],1),og): return None
    return True

def detect_nonzero_recolor(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    tc=None
    for p in pairs:
        ig=np.array(p['input']); og=np.array(p['output'])
        if not np.all(og[ig==0]==0): return None
        nb=og[ig!=0]
        if nb.size==0: continue
        u=np.unique(nb)
        if len(u)!=1: return None
        if tc is None: tc=u[0]
        elif tc!=u[0]: return None
    return tc

def detect_bg_recolor(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    tc=None
    for p in pairs:
        ig=np.array(p['input']); og=np.array(p['output'])
        if not np.array_equal(og[ig!=0],ig[ig!=0]): return None
        bo=og[ig==0]
        if bo.size==0: continue
        u=np.unique(bo)
        if len(u)!=1: return None
        if tc is None: tc=u[0]
        elif tc!=u[0]: return None
    return tc

def detect_fixed_largest_bbox_crop(pairs):
    boxes=[]
    for p in pairs:
        ig=np.array(p['input']); og=np.array(p['output'])
        comps=_connected_components_nonbg(ig)
        if not comps: return None
        biggest=max(comps,key=len)
        rs=[r for r,_ in biggest]; cs=[c for _,c in biggest]
        r0,r1=min(rs),max(rs); c0,c1=min(cs),max(cs)
        if og.shape!=(r1-r0+1,c1-c0+1): return None
        if not np.array_equal(ig[r0:r1+1,c0:c1+1],og): return None
        boxes.append((r0,c0,r1-r0+1,c1-c0+1))
    if len(set(boxes))==1: return boxes[0]
    return None

def detect_fixed_nonbg_bbox(pairs):
    boxes=[]
    for p in pairs:
        ig=np.array(p['input']); og=np.array(p['output'])
        mask=ig!=0
        if not mask.any(): return None
        rs,cs=np.where(mask)
        r0,r1=rs.min(),rs.max(); c0,c1=cs.min(),cs.max()
        crop=ig[r0:r1+1,c0:c1+1]
        if crop.shape!=og.shape or not np.array_equal(crop,og): return None
        boxes.append((r0,c0,r1-r0+1,c1-c0+1))
    if len(set(boxes))==1: return boxes[0]
    return None

print('All detectors loaded.')

In [ ]:
def detect_dilate_3x3(pairs):
    """Each cell becomes max (per channel) over its 3x3 neighborhood."""
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig = np.array(p['input']); og = np.array(p['output'])
        h, w = ig.shape
        t = grid_to_tensor(ig.tolist())[0]   # [C, 30, 30]
        dilated = np.zeros_like(t)
        for c in range(1, C):
            for r in range(h):
                for cc in range(w):
                    for dr in range(-1, 2):
                        for dc in range(-1, 2):
                            rr, ccc = r+dr, cc+dc
                            if 0 <= rr < h and 0 <= ccc < w:
                                if t[c, rr, ccc] > dilated[c, r, cc]:
                                    dilated[c, r, cc] = t[c, rr, ccc]
        out = np.zeros((h, w), dtype=int)
        for r in range(h):
            for cc in range(w):
                vals = dilated[1:, r, cc]
                if vals.max() > 0:
                    out[r, cc] = 1 + int(vals.argmax())
        if not np.array_equal(out, og): return False
    return True

print('detect_dilate_3x3 loaded.')
def make_dilate_3x3_onnx(kernel_size=3):
    """Per-channel MaxPool 3x3 — dilates each color into its 8-neighborhood."""
    X = oh.make_tensor_value_info('input',  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info('output', TensorProto.FLOAT, [1, C, H, W])
    nodes  = _slice_channel_node('input', 'nonbg_ch', 1, C, 'nb')
    pad = kernel_size // 2
    nodes += [oh.make_node('MaxPool', ['nonbg_ch'], ['dilated'],
                            kernel_shape=[kernel_size, kernel_size],
                            pads=[pad, pad, pad, pad], strides=[1, 1])]
    nodes += [oh.make_node('ReduceMax', ['dilated'], ['any_d'], axes=[1], keepdims=1)]
    one_t = np.ones((1, 1, 1, 1), dtype=np.float32)
    nodes += [
        _const_node('dil_one', one_t),
        oh.make_node('Sub', ['dil_one', 'any_d'], ['new_bg']),
    ]
    nodes += [oh.make_node('Concat', ['new_bg', 'dilated'], ['output'], axis=1)]
    graph = oh.make_graph(nodes, 'dilate', [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid('', 17)])
    model.ir_version = 8
    return model

print('detect_dilate_3x3 + make_dilate_3x3_onnx loaded.')

In [ ]:
def build_rotation_gather(angle, ih, iw):
    k=angle//90; idx=np.full(HW,SAFE_PAD_IDX,dtype=np.int64)
    if k==1:
        for r in range(iw):
            for c in range(ih): idx[r*W+c]=c*W+(iw-1-r)
    elif k==2:
        for r in range(ih):
            for c in range(iw): idx[r*W+c]=(ih-1-r)*W+(iw-1-c)
    elif k==3:
        for r in range(iw):
            for c in range(ih): idx[r*W+c]=(ih-1-c)*W+r
    return idx

def build_flip_gather(direction, ih, iw):
    idx=np.full(HW,SAFE_PAD_IDX,dtype=np.int64)
    for r in range(ih):
        for c in range(iw):
            if direction=='vertical':   idx[r*W+c]=(ih-1-r)*W+c
            elif direction=='horizontal': idx[r*W+c]=r*W+(iw-1-c)
            else: idx[r*W+c]=(ih-1-r)*W+(iw-1-c)
    return idx

def build_transpose_gather(ih, iw):
    idx=np.full(HW,SAFE_PAD_IDX,dtype=np.int64)
    for r in range(iw):
        for c in range(ih): idx[r*W+c]=c*W+r
    return idx

def build_anti_transpose_gather(ih, iw):
    idx=np.full(HW,SAFE_PAD_IDX,dtype=np.int64)
    for r in range(iw):
        for c in range(ih):
            sr=ih-1-c; sc=iw-1-r
            if 0<=sr<H and 0<=sc<W: idx[r*W+c]=sr*W+sc
    return idx

def build_scale_gather(factor, ih, iw):
    idx=np.full(HW,SAFE_PAD_IDX,dtype=np.int64)
    for r in range(min(H,ih*factor)):
        for c in range(min(W,iw*factor)): idx[r*W+c]=(r//factor)*W+(c//factor)
    return idx

def build_downscale_gather(factor, ih, iw):
    idx=np.full(HW,SAFE_PAD_IDX,dtype=np.int64)
    oh_=ih//factor; ow_=iw//factor
    for r in range(oh_):
        for c in range(ow_): idx[r*W+c]=(r*factor)*W+(c*factor)
    return idx

def build_tile_gather(tr, tc, ih, iw):
    idx=np.full(HW,SAFE_PAD_IDX,dtype=np.int64)
    for r in range(min(H,ih*tr)):
        for c in range(min(W,iw*tc)): idx[r*W+c]=(r%ih)*W+(c%iw)
    return idx

def build_h_sym_complete_gathers(ih, iw):
    idx_id=np.full(HW,SAFE_PAD_IDX,dtype=np.int64)
    idx_m=np.full(HW,SAFE_PAD_IDX,dtype=np.int64)
    for r in range(ih):
        for c in range(iw): idx_id[r*W+c]=r*W+c; idx_m[r*W+c]=r*W+(iw-1-c)
    return [idx_id,idx_m]

def build_v_sym_complete_gathers(ih, iw):
    idx_id=np.full(HW,SAFE_PAD_IDX,dtype=np.int64)
    idx_m=np.full(HW,SAFE_PAD_IDX,dtype=np.int64)
    for r in range(ih):
        for c in range(iw): idx_id[r*W+c]=r*W+c; idx_m[r*W+c]=(ih-1-r)*W+c
    return [idx_id,idx_m]

def build_both_sym_complete_gathers(ih, iw):
    idxs=[]
    for t in range(4):
        idx=np.full(HW,SAFE_PAD_IDX,dtype=np.int64)
        for r in range(ih):
            for c in range(iw):
                if t==0: sr,sc=r,c
                elif t==1: sr,sc=r,iw-1-c
                elif t==2: sr,sc=ih-1-r,c
                else: sr,sc=ih-1-r,iw-1-c
                idx[r*W+c]=sr*W+sc
        idxs.append(idx)
    return idxs

def build_diag_sym_complete_gathers(ih, iw):
    if ih!=iw: return None
    idx_id=np.full(HW,SAFE_PAD_IDX,dtype=np.int64)
    idx_m=np.full(HW,SAFE_PAD_IDX,dtype=np.int64)
    for r in range(ih):
        for c in range(iw): idx_id[r*W+c]=r*W+c; idx_m[r*W+c]=c*W+r
    return [idx_id,idx_m]

def can_use_channel_gather_for_cmap(cmap, train_pairs):
    if cmap is None: return False
    changed={s:d for s,d in cmap.items() if s!=d}
    if not changed: return False
    if set(changed.keys())==set(changed.values()): return True
    return False

def build_channel_gather_indices(cmap):
    gi=np.arange(C,dtype=np.int32)
    for src,dst in cmap.items():
        if src!=dst: gi[dst]=src
    return gi

def build_color_map_weight(color_map):
    w=np.zeros((C,C,1,1),dtype=np.float32)
    for src,dst in color_map.items():
        if 0<=src<C and 0<=dst<C: w[dst,src,0,0]=1.0
    for ch in range(C):
        if ch not in color_map: w[ch,ch,0,0]=1.0
    return w

def build_nonzero_recolor_weight(tc):
    w=np.zeros((C,C,1,1),dtype=np.float32); w[0,0,0,0]=1.0
    for s in range(1,C): w[tc,s,0,0]=1.0
    return w

def build_bg_recolor_weight(tc):
    w=np.zeros((C,C,1,1),dtype=np.float32); w[tc,0,0,0]=1.0
    for s in range(1,C): w[s,s,0,0]=1.0
    return w

def try_spatial_zero_ambiguity(all_pairs):
    for p in all_pairs:
        ig=np.array(p['input']); og=np.array(p['output'])
        if ig.shape[0]>H or ig.shape[1]>W or og.shape[0]>H or og.shape[1]>W: return None
    inputs  = np.stack([grid_to_tensor(p['input'])[0].reshape(C,-1)  for p in all_pairs])
    outputs = np.stack([grid_to_tensor(p['output'])[0].reshape(C,-1) for p in all_pairs])
    lookup={}
    for si in range(HW): lookup.setdefault(inputs[:,:,si].tobytes(),[]).append(si)
    gi=np.full(HW,SAFE_PAD_IDX,dtype=np.int64)
    for oi in range(HW):
        sa=outputs[:,:,oi]
        if np.all(sa==0): continue
        cands=lookup.get(sa.tobytes())
        if not cands or len(cands)>1: return None
        gi[oi]=cands[0]
    return make_gather_onnx(gi)

def try_spatial_relaxed(all_pairs, max_ambig=12):
    for p in all_pairs:
        ig=np.array(p['input']); og=np.array(p['output'])
        if ig.shape[0]>H or ig.shape[1]>W or og.shape[0]>H or og.shape[1]>W: return None
    inputs  = np.stack([grid_to_tensor(p['input'])[0].reshape(C,-1)  for p in all_pairs])
    outputs = np.stack([grid_to_tensor(p['output'])[0].reshape(C,-1) for p in all_pairs])
    lookup={}
    for si in range(HW): lookup.setdefault(inputs[:,:,si].tobytes(),[]).append(si)
    gi=np.full(HW,SAFE_PAD_IDX,dtype=np.int64); ambig=[]
    for oi in range(HW):
        sa=outputs[:,:,oi]
        if np.all(sa==0): continue
        cands=lookup.get(sa.tobytes())
        if not cands: return None
        gi[oi]=cands[0]
        if len(cands)>1: ambig.append((oi,cands))
    if len(ambig)>max_ambig: return None
    m=make_gather_onnx(gi)
    if check_model_correct(m,all_pairs): return m
    if len(ambig)>8: return None
    for combo in iprod(*(c for _,c in ambig)):
        for idx,(oi,_) in enumerate(ambig): gi[oi]=combo[idx]
        m=make_gather_onnx(gi)
        if check_model_correct(m,all_pairs): return m
    return None

print('Gather builders + spatial solvers loaded.')

In [ ]:
def _get_torch_device():
    try:
        import torch
        return torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    except ImportError:
        return None

def try_learned_conv_fast(pairs, all_pairs, kernel_size=1, max_steps=1000, lr=0.02, use_bias=False):
    try:
        import torch, torch.nn as nn
    except: return None
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    inp = torch.tensor(np.stack([grid_to_tensor(p['input'])[0]  for p in pairs]), dtype=torch.float32, device=device)
    out = torch.tensor(np.stack([grid_to_tensor(p['output'])[0] for p in pairs]), dtype=torch.float32, device=device)
    pad = kernel_size // 2
    conv = nn.Conv2d(C, C, kernel_size=kernel_size, padding=pad, bias=use_bias).to(device)
    nn.init.zeros_(conv.weight)
    if use_bias: nn.init.zeros_(conv.bias)
    for i in range(C): conv.weight.data[i, i, pad, pad] = 1.0
    optimizer = torch.optim.Adam(conv.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=300, gamma=0.5)
    best_loss = float('inf'); best_state = None; plateau = 0
    for step in range(max_steps):
        optimizer.zero_grad()
        loss = nn.functional.mse_loss(conv(inp), out)
        loss.backward(); optimizer.step(); scheduler.step()
        l = loss.item()
        if l < best_loss - 1e-7: best_loss = l; best_state = copy.deepcopy(conv.state_dict()); plateau = 0
        else: plateau += 1
        # CHANGE 6: Earlier exits
        if step == 100 and best_loss > 0.8: return None
        if step == 250 and best_loss > 0.5: return None
        if step == 500 and best_loss > 0.15: return None
        if plateau > 200 and best_loss > 0.01: return None
        if best_loss < 1e-6: break
    if best_loss > 0.05: return None
    conv.load_state_dict(best_state)
    w = conv.weight.detach().cpu().numpy()
    b = conv.bias.detach().cpu().numpy() if use_bias else None
    best_model = None; best_cost = float('inf')
    for w_try in [np.round(w), np.where(np.abs(w) < 0.01, 0, w), w]:
        b_try = np.round(b) if b is not None else None
        m = make_conv_onnx(w_try, bias=b_try, kernel_size=kernel_size)
        if check_model_correct(m, all_pairs):
            c = estimate_model_cost(m)
            if c < best_cost: best_cost = c; best_model = m
    return best_model

def try_two_layer_conv_fast(pairs, all_pairs, ks1=3, ks2=1, hidden=16,
                             max_steps=2000, lr=0.015, deadline=None):
    try:
        import torch, torch.nn as nn
    except: return None
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    inp = torch.tensor(np.stack([grid_to_tensor(p['input'])[0]  for p in pairs]), dtype=torch.float32, device=device)
    out = torch.tensor(np.stack([grid_to_tensor(p['output'])[0] for p in pairs]), dtype=torch.float32, device=device)
    pad1 = ks1//2; pad2 = ks2//2
    torch.manual_seed(0)
    net = nn.Sequential(
        nn.Conv2d(C, hidden, kernel_size=ks1, padding=pad1, bias=True),
        nn.ReLU(),
        nn.Conv2d(hidden, C, kernel_size=ks2, padding=pad2, bias=False),
    ).to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_steps)
    best_loss = float('inf'); best_state = None; plateau = 0
    for step in range(max_steps):
        if deadline is not None and time.time() > deadline: break
        optimizer.zero_grad()
        loss = nn.functional.mse_loss(net(inp), out)
        loss.backward(); optimizer.step(); scheduler.step()
        l = loss.item()
        if l < best_loss - 1e-7: best_loss = l; best_state = copy.deepcopy(net.state_dict()); plateau = 0
        else: plateau += 1
        if step == 150 and best_loss > 0.9: return None   # stuck early
        if step == 300 and best_loss > 0.5: return None
        if step == 700 and best_loss > 0.15: return None
        if plateau > 250 and best_loss > 0.005: return None
        if best_loss < 1e-6: break
    if best_loss > 0.05 or best_state is None: return None
    net.load_state_dict(best_state)
    w1 = np.where(np.abs(v := net[0].weight.detach().cpu().numpy()) < 0.01, 0, v)
    b1 = net[0].bias.detach().cpu().numpy()
    w2 = np.where(np.abs(v := net[2].weight.detach().cpu().numpy()) < 0.01, 0, v)
    X  = oh.make_tensor_value_info('input',  TensorProto.FLOAT, [1, C, H, W])
    Y  = oh.make_tensor_value_info('output', TensorProto.FLOAT, [1, C, H, W])
    nodes = [
        oh.make_node('Constant', [], ['w1'], value=onh.from_array(w1.astype(np.float32), name='w1')),
        oh.make_node('Constant', [], ['b1'], value=onh.from_array(b1.astype(np.float32), name='b1')),
        oh.make_node('Conv', ['input','w1','b1'], ['c1'], kernel_shape=[ks1,ks1], pads=[pad1]*4),
        oh.make_node('Relu', ['c1'], ['r1']),
        oh.make_node('Constant', [], ['w2'], value=onh.from_array(w2.astype(np.float32), name='w2')),
        oh.make_node('Conv', ['r1','w2'], ['output'], kernel_shape=[ks2,ks2], pads=[pad2]*4),
    ]
    graph = oh.make_graph(nodes, 'two_layer', [X], [Y])
    m = oh.make_model(graph, opset_imports=[oh.make_opsetid('', 17)]); m.ir_version = 8
    return m if check_model_correct(m, all_pairs) else None

def try_three_layer_conv(pairs, all_pairs, max_steps=2500, deadline=None):
    try:
        import torch, torch.nn as nn
    except: return None
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    inp = torch.tensor(np.stack([grid_to_tensor(p['input'])[0]  for p in pairs]), dtype=torch.float32, device=device)
    out = torch.tensor(np.stack([grid_to_tensor(p['output'])[0] for p in pairs]), dtype=torch.float32, device=device)
    class Net(nn.Module):
        def __init__(s):
            super().__init__()
            s.c1=nn.Conv2d(C,16,3,padding=1,bias=True)
            s.c2=nn.Conv2d(16,16,3,padding=1,bias=True)
            s.c3=nn.Conv2d(16,C,1,bias=True)
        def forward(s,x): return s.c3(torch.relu(s.c2(torch.relu(s.c1(x)))))
    torch.manual_seed(0)
    net = Net().to(device); optimizer = torch.optim.Adam(net.parameters(), lr=0.005)
    best_loss = float('inf'); best_state = None
    for step in range(max_steps):
        if deadline is not None and time.time() > deadline: break
        optimizer.zero_grad()
        loss = nn.functional.mse_loss(net(inp), out)
        loss.backward(); optimizer.step()
        if loss.item() < best_loss: best_loss = loss.item(); best_state = copy.deepcopy(net.state_dict())
        if step == 500 and best_loss > 0.3: return None
        if best_loss < 1e-6: break
    if best_loss > 0.05 or best_state is None: return None
    net.load_state_dict(best_state)
    X = oh.make_tensor_value_info('input',  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info('output', TensorProto.FLOAT, [1, C, H, W])
    def cpu(t): return t.detach().cpu().numpy().astype(np.float32)
    nodes = [
        oh.make_node('Constant',[], ['w1'], value=onh.from_array(cpu(net.c1.weight),name='w1')),
        oh.make_node('Constant',[], ['b1'], value=onh.from_array(cpu(net.c1.bias),  name='b1')),
        oh.make_node('Conv',['input','w1','b1'],['c1'],kernel_shape=[3,3],pads=[1,1,1,1]),
        oh.make_node('Relu',['c1'],['r1']),
        oh.make_node('Constant',[], ['w2'], value=onh.from_array(cpu(net.c2.weight),name='w2')),
        oh.make_node('Constant',[], ['b2'], value=onh.from_array(cpu(net.c2.bias),  name='b2')),
        oh.make_node('Conv',['r1','w2','b2'],['c2'],kernel_shape=[3,3],pads=[1,1,1,1]),
        oh.make_node('Relu',['c2'],['r2']),
        oh.make_node('Constant',[], ['w3'], value=onh.from_array(cpu(net.c3.weight),name='w3')),
        oh.make_node('Constant',[], ['b3'], value=onh.from_array(cpu(net.c3.bias),  name='b3')),
        oh.make_node('Conv',['r2','w3','b3'],['output'],kernel_shape=[1,1],pads=[0,0,0,0]),
    ]
    graph = oh.make_graph(nodes, 'three_layer', [X], [Y])
    m = oh.make_model(graph, opset_imports=[oh.make_opsetid('', 17)]); m.ir_version = 8
    return m if check_model_correct(m, all_pairs) else None

print('Learned conv (CHANGE 6 applied) loaded.')

In [ ]:
def analyze_transformation(pairs):
    info = {}
    in_sizes  = [np.array(p['input']).shape  for p in pairs]
    out_sizes = [np.array(p['output']).shape for p in pairs]
    info['in_sizes']  = in_sizes
    info['out_sizes'] = out_sizes
    info['same_size'] = all(i==o for i,o in zip(in_sizes,out_sizes))
    info['const_output'] = len(set(json.dumps(p['output']) for p in pairs)) == 1
    info['all_same_in_size'] = len(set(in_sizes)) == 1
    info['identity'] = info['same_size'] and all(
        np.array_equal(np.array(p['input']),np.array(p['output'])) for p in pairs)
    if info['same_size']:
        m={}; valid=True
        for p in pairs:
            for a,b in zip(np.array(p['input']).flat, np.array(p['output']).flat):
                if a in m and m[a]!=b: valid=False; break
                m[a]=b
            if not valid: break
        info['color_map'] = m if valid and any(k!=v for k,v in m.items()) else None
    else:
        info['color_map'] = None
    return info

print('Analysis loaded.')

In [ ]:
def solve_task(task_data, task_name='', time_budget=15.0):
    train_pairs = task_data.get('train', [])
    test_pairs  = task_data.get('test',  [])
    arc_gen     = task_data.get('arc-gen', [])[:20]
    if not train_pairs: return make_identity_onnx()

    all_pairs = train_pairs + test_pairs + arc_gen
    info = analyze_transformation(train_pairs)
    best_model = None; best_cost = float('inf')
    t_start = time.time()

    def try_model(model, label=''):
        nonlocal best_model, best_cost
        if model is None: return False
        if check_model_correct(model, all_pairs):
            cost = estimate_model_cost(model)
            if cost < best_cost:
                best_cost = cost; best_model = model
            return True
        return False

    def elapsed():  return time.time() - t_start
    def deadline(): return t_start + time_budget

    # ---- IDENTITY ----
    if info.get('identity'):
        try_model(make_identity_onnx(), 'identity')
        if best_model: return best_model

    has_spatial = (
        detect_rotation(train_pairs) is not None or
        detect_flip(train_pairs)     is not None or
        detect_transpose(train_pairs) or
        detect_anti_transpose(train_pairs)
    )

    # ---- COLOR MAPS ----
    if info.get('color_map') and info['same_size']:
        cmap = info['color_map']
        if can_use_channel_gather_for_cmap(cmap, train_pairs) and not has_spatial:
            try_model(make_channel_gather_onnx(build_channel_gather_indices(cmap)), 'cmap_ch')
        try_model(make_conv1x1_onnx(build_color_map_weight(cmap)), 'cmap_conv')

    tc = detect_nonzero_recolor(train_pairs)
    if tc is not None: try_model(make_conv1x1_onnx(build_nonzero_recolor_weight(tc)), 'nz_recolor')

    tc = detect_bg_recolor(train_pairs)
    if tc is not None: try_model(make_conv1x1_onnx(build_bg_recolor_weight(tc)), 'bg_recolor')

    repl = detect_color_replace(train_pairs)
    if repl:
        cmap = {i:i for i in range(C)}; cmap.update(repl)
        if can_use_channel_gather_for_cmap(cmap, train_pairs) and not has_spatial:
            try_model(make_channel_gather_onnx(build_channel_gather_indices(cmap)), 'repl_ch')
        try_model(make_conv1x1_onnx(build_color_map_weight(cmap)), 'repl_conv')

    sw = detect_color_swap(train_pairs)
    if sw and not has_spatial:
        a,b = sw; cmap = {i:i for i in range(C)}; cmap[a]=b; cmap[b]=a
        try_model(make_channel_gather_onnx(build_channel_gather_indices(cmap)), 'swap_ch')

    cf = detect_color_filter(train_pairs)
    if cf and info['same_size']:
        cmap = {c_:0 for c_ in cf}; cmap.update({c_:c_ for c_ in range(C) if c_ not in cf})
        if can_use_channel_gather_for_cmap(cmap, train_pairs):
            try_model(make_channel_gather_onnx(build_channel_gather_indices(cmap)), 'cf_ch')
        try_model(make_conv1x1_onnx(build_color_map_weight(cmap)), 'cf_conv')

    # ---- ROTATIONS / FLIPS / TRANSPOSE ----
    rot = detect_rotation(train_pairs)
    in_shapes = set(np.array(p['input']).shape for p in train_pairs)
    if rot and len(in_shapes) == 1:
        ih, iw = list(in_shapes)[0]
        try_model(make_gather_onnx(build_rotation_gather(rot, ih, iw)), f'rot_{rot}')
        if rot == 90:  try_model(make_cheap_rot90ccw_framed(ih, iw), 'rot90_cheap')
        elif rot==180: try_model(make_cheap_rot180_framed(ih, iw),   'rot180_cheap')
        elif rot==270: try_model(make_cheap_rot90cw_framed(ih, iw),  'rot270_cheap')

    flip = detect_flip(train_pairs)
    if flip and len(in_shapes) == 1:
        ih, iw = list(in_shapes)[0]
        try_model(make_gather_onnx(build_flip_gather(flip, ih, iw)), f'flip_{flip}')
        if flip=='vertical':   try_model(make_cheap_flip_h_framed(ih, iw), 'flip_v_cheap')
        elif flip=='horizontal': try_model(make_cheap_flip_w_framed(ih, iw), 'flip_h_cheap')
        elif flip=='both':     try_model(make_cheap_rot180_framed(ih, iw),  'flip_b_cheap')

    if detect_transpose(train_pairs) and len(in_shapes)==1:
        ih, iw = list(in_shapes)[0]
        try_model(make_cheap_transpose_framed(ih, iw), 'tr_cheap')
    if detect_anti_transpose(train_pairs) and len(in_shapes)==1:
        ih, iw = list(in_shapes)[0]
        try_model(make_cheap_anti_transpose_framed(ih, iw), 'atr_cheap')

    # ---- SPATIAL + COLOR COMPOSITE ----
    def _try_spatial_plus_color(spatial_gi, label):
        try:
            cmap_accum = None
            for p in train_pairs:
                ig = np.array(p['input']); og = np.array(p['output'])
                ih2, iw2 = ig.shape
                if ih2>H or iw2>W: return
                flat = np.zeros((H,W),dtype=np.int32); flat[:ih2,:iw2]=ig
                flat_full = flat.flatten()
                mapped = np.zeros(HW,dtype=np.int32)
                for i in range(HW):
                    si=int(spatial_gi[i])
                    mapped[i]=flat_full[si] if si!=SAFE_PAD_IDX else 0
                oh2,ow2=og.shape
                if oh2>H or ow2>W: return
                region=mapped.reshape(H,W)[:oh2,:ow2]
                if region.shape!=og.shape: return
                cmap_local={}
                for a,b in zip(region.flatten(),og.flatten()):
                    a,b=int(a),int(b)
                    if a in cmap_local and cmap_local[a]!=b: return
                    cmap_local[a]=b
                if cmap_accum is None: cmap_accum=dict(cmap_local)
                else:
                    for k,v in cmap_local.items():
                        if k in cmap_accum and cmap_accum[k]!=v: return
                        cmap_accum[k]=v
            if cmap_accum is None: return
            cmap_full={i:i for i in range(C)}; cmap_full.update(cmap_accum)
            if all(k==v for k,v in cmap_full.items()): return
            if can_use_channel_gather_for_cmap(cmap_full,train_pairs):
                try_model(make_spatial_then_channel_gather_onnx(
                    spatial_gi,build_channel_gather_indices(cmap_full)), f'{label}_ch_cm')
            try_model(make_gather_then_conv1x1_onnx(
                spatial_gi,build_color_map_weight(cmap_full)), f'{label}_cm')
        except: return

    if len(in_shapes)==1:
        ih,iw=list(in_shapes)[0]
        for angle in [90,180,270]:
            _try_spatial_plus_color(build_rotation_gather(angle,ih,iw),f'rot{angle}')
        for direction in ['vertical','horizontal','both']:
            _try_spatial_plus_color(build_flip_gather(direction,ih,iw),f'flip_{direction}')
        if ih<=W and iw<=H:
            _try_spatial_plus_color(build_transpose_gather(ih,iw),'transpose')
            _try_spatial_plus_color(build_anti_transpose_gather(ih,iw),'anti_transpose')
        for factor in [2,3]:
            if ih*factor<=H and iw*factor<=W:
                _try_spatial_plus_color(build_scale_gather(factor,ih,iw),f'scale{factor}')
            if ih%factor==0 and iw%factor==0:
                _try_spatial_plus_color(build_downscale_gather(factor,ih,iw),f'ds{factor}')

    # ---- CROP / TILE / SCALE ----
    crop = detect_crop(train_pairs)
    if crop:
        r0,c0,oh_,ow_=crop; try_model(make_cheap_crop(r0,c0,oh_,ow_),'crop')

    tile = detect_tile(train_pairs)
    if tile:
        tr,tc=tile; ih,iw=np.array(train_pairs[0]['input']).shape
        if ih*tr<=H and iw*tc<=W: try_model(make_cheap_tile(tr,tc,ih,iw),'tile')

    sc = detect_scale(train_pairs)
    if sc:
        ih,iw=np.array(train_pairs[0]['input']).shape
        if ih*sc<=H and iw*sc<=W: try_model(make_cheap_upscale(sc,ih,iw),'scale')

    ds = detect_downscale(train_pairs)
    if ds:
        ih,iw=np.array(train_pairs[0]['input']).shape
        try_model(make_cheap_downscale(ds,ih,iw),'downscale')

    perm = detect_pixel_permutation(train_pairs)
    if perm is not None: try_model(make_gather_onnx(perm),'perm')

    # Mirror/concat
    if len(in_shapes)==1:
        ih,iw=list(in_shapes)[0]
        if detect_mirror_h_concat(train_pairs) and iw*2<=W: try_model(make_cheap_concat_flip_w(ih,iw),'mirror_h')
        if detect_mirror_v_concat(train_pairs) and ih*2<=H: try_model(make_cheap_concat_flip_h(ih,iw),'mirror_v')
        if detect_self_concat_h(train_pairs) and iw*2<=W: try_model(make_cheap_self_concat_w(ih,iw),'self_h')
        if detect_self_concat_v(train_pairs) and ih*2<=H: try_model(make_cheap_self_concat_h(ih,iw),'self_v')
        qm=detect_quad_mirror(train_pairs)
        if qm and ih*2<=H and iw*2<=W: try_model(make_cheap_quadrant_mirror(ih,iw),'quad')
        if detect_concat_rot180_w(train_pairs) and iw*2<=W: try_model(make_cheap_concat_rot180_w(ih,iw),'cat_r180')

    trans=detect_translation(train_pairs)
    if trans and len(in_shapes)==1:
        dr,dc=trans; ih,iw=list(in_shapes)[0]
        m=make_cheap_framed_translation(ih,iw,dr,dc)
        if m: try_model(m,'translate')

    if len(in_shapes)==1:
        ih,iw=list(in_shapes)[0]
        rb=detect_row_broadcast(train_pairs)
        if rb is not None: try_model(make_cheap_row_broadcast(ih,iw,rb),'rb')
        cb=detect_col_broadcast(train_pairs)
        if cb is not None: try_model(make_cheap_col_broadcast(ih,iw,cb),'cb')

    ba=detect_border_add(train_pairs)
    if ba and len(in_shapes)==1:
        b,fill=ba; ih,iw=list(in_shapes)[0]
        if ih+2*b<=H and iw+2*b<=W:
            m=make_cheap_border_add(ih,iw,b,fill)
            if m: try_model(m,'border')

    ccp=detect_crop_plus_color(train_pairs)
    if ccp:
        r0,c0,oh_,ow_,cm=ccp
        try_model(make_cheap_crop_then_color(r0,c0,oh_,ow_,cm),'crop_color')

    lob=detect_fixed_largest_bbox_crop(train_pairs)
    if lob:
        r0,c0,oh_,ow_=lob; try_model(make_cheap_crop(r0,c0,oh_,ow_),'largest_bbox')
    nbb=detect_fixed_nonbg_bbox(train_pairs)
    if nbb:
        r0,c0,oh_,ow_=nbb; try_model(make_cheap_crop(r0,c0,oh_,ow_),'nonbg_bbox')

    # Symmetry completion
    if info['same_size'] and info['all_same_in_size']:
        ih,iw=np.array(train_pairs[0]['input']).shape
        if detect_h_symmetry_complete(train_pairs):
            try_model(make_symmetry_completion_onnx(build_h_sym_complete_gathers(ih,iw)),'h_sym')
        if detect_v_symmetry_complete(train_pairs):
            try_model(make_symmetry_completion_onnx(build_v_sym_complete_gathers(ih,iw)),'v_sym')
        if detect_both_symmetry_complete(train_pairs):
            try_model(make_symmetry_completion_onnx(build_both_sym_complete_gathers(ih,iw)),'both_sym')
        if ih==iw and detect_diag_symmetry_complete(train_pairs):
            g=build_diag_sym_complete_gathers(ih,iw)
            if g: try_model(make_symmetry_completion_onnx(g),'diag_sym')

    # V11 content-dependent primitives
    bf=detect_bbox_fill(train_pairs)
    if bf is not None: try_model(make_bbox_fill_onnx(bf),f'bbox_fill_{bf}')
    hf=detect_hole_fill(train_pairs)
    if hf is not None: try_model(make_hole_fill_onnx(hf,num_iters=14),f'hole_fill_{hf}')
    if detect_outline_extract(train_pairs): try_model(make_outline_onnx(),'outline')
    if detect_keep_largest_color(train_pairs): try_model(make_keep_largest_color_onnx(),'keep_largest')
    if detect_keep_smallest_color(train_pairs): try_model(make_keep_smallest_color_onnx(),'keep_smallest')
        
    in_sh = set(np.array(p['input']).shape for p in train_pairs)
    if len(in_sh) == 1:
        ih_x, iw_x = list(in_sh)[0]

        # Period row tiling
        prt = detect_period_row_tiling(train_pairs)
        if prt:
            period, in_h, in_w, out_h = prt
            if out_h <= H:
                try_model(make_period_row_tile_onnx(period, in_h, in_w, out_h), f'prt_{period}')

        # Period col tiling
        pct = detect_period_col_tiling(train_pairs)
        if pct:
            period, in_h, in_w, out_w = pct
            if out_w <= W:
                try_model(make_period_col_tile_onnx(period, in_h, in_w, out_w), f'pct_{period}')

        # Frame border
        fb = detect_frame_border(train_pairs)
        if fb is not None:
            try_model(make_frame_border_onnx(fb, ih_x, iw_x), f'frame_border_{fb}')

        # Fixed row select + tile
        frs = detect_fixed_row_select(train_pairs)
        if frs is not None:
            row_idx, out_h = frs
            if out_h <= H:
                try_model(make_row_select_tile_onnx(row_idx, iw_x, out_h), f'row_select_{row_idx}')

    # Bbox crop then tile
    bbt = detect_bbox_then_tile(train_pairs)
    if bbt:
        r0, c0, ch, cw, tr, tc = bbt
        if ch * tr <= H and cw * tc <= W:
            try_model(make_bbox_tile_onnx(r0, c0, ch, cw, tr, tc), f'bbox_tile_{tr}x{tc}')

    # Crop then scale
    cts = detect_crop_then_scale(train_pairs)
    if cts:
        r0, c0, ch, cw, factor = cts
        if ch * factor <= H and cw * factor <= W:
            try_model(make_crop_scale_onnx(r0, c0, ch, cw, factor), f'crop_scale_{factor}')

    # ---- SPATIAL SOLVERS ----
    m = try_spatial_zero_ambiguity(all_pairs)
    if m: try_model(m,'spatial_0')

    if best_model is None and elapsed() < time_budget * 0.6:
        m = try_spatial_relaxed(all_pairs, max_ambig=12)
        if m: try_model(m,'spatial_r')

    if best_model is not None: return best_model

    conv_configs_1layer = [
        dict(kernel_size=1,  max_steps=400,  lr=0.02,  use_bias=False),
        dict(kernel_size=3,  max_steps=400,  lr=0.015, use_bias=False),
    ]
    for cfg in conv_configs_1layer:
        if elapsed() > time_budget * 0.5: break
        m = try_learned_conv_fast(train_pairs, all_pairs, **cfg)
        if try_model(m, f'lconv_{cfg["kernel_size"]}'): break

    if best_model is not None: return best_model

    two_layer_configs = [
        dict(ks1=3, ks2=1, hidden=16, max_steps=600, lr=0.015),
        dict(ks1=3, ks2=1, hidden=24, max_steps=600, lr=0.012),
        dict(ks1=3, ks2=3, hidden=16, max_steps=600, lr=0.015),
    ]
    for cfg in two_layer_configs:
        if elapsed() > time_budget * 0.85: break
        m = try_two_layer_conv_fast(train_pairs, all_pairs, deadline=deadline(), **cfg)
        if try_model(m, f'l2_{cfg["ks1"]}_{cfg["hidden"]}'): break

    if best_model is not None: return best_model


    if best_model is not None: return best_model

    # Constant output (last resort)
    if info.get('const_output'):
        try_model(make_const_onnx(grid_to_tensor(train_pairs[0]['output'])),'const')

    return best_model

print('solve_task (CHANGE 5+6 applied) loaded.')

In [ ]:
def save_model(model, path):
    with open(path, 'wb') as f: f.write(model.SerializeToString())

def find_task_files():
    for d in [TASK_DIR, Path('/kaggle/input'), Path('.')]:
        if d.exists():
            files = sorted(d.glob('task*.json'))
            if files: return files
    files = []
    root = '/kaggle/input' if Path('/kaggle/input').exists() else '.'
    for r, _, fs in os.walk(root):
        for f in fs:
            if f.startswith('task') and f.endswith('.json'):
                files.append(Path(r) / f)
    return sorted(files)

task_files = find_task_files()
if not task_files:
    print('ERROR: no tasks found'); sys.exit(1)
print(f'Found {len(task_files)} tasks')

solved = []; onnx_files = []; total_score = 0.0
start = time.time()

for tp in task_files:
    tn   = tp.stem
    tnum = tn.replace('task', '')
    fname = f'task{tnum}.onnx'

    try: td = load_task(tp)
    except Exception as e:
        print(f'{tn}: load err {e}'); continue

    # Skip tasks already covered by floor — solver never improved them last run
    if fname in floor_solutions:
        continue

    budget = 15.0  # no-floor tasks only

    t0 = time.time()
    try: model = solve_task(td, tn, time_budget=budget)
    except Exception as e:
        print(f'{tn}: ERR {e}'); traceback.print_exc(); continue
    dt = time.time() - t0

    if model is None:
        continue

    op = OUTPUT_DIR / fname
    save_model(model, op); onnx_files.append(op)
    cost  = estimate_model_cost(model)
    score = max(1, 25 - math.log(max(1, cost)))
    total_score += score; solved.append(tn)
    print(f'  {tn}: cost={cost:>10} score={score:.1f} ({dt:.1f}s)')

total = time.time() - start
print(f'\n{len(solved)}/{len(task_files)} solved by our solver, score={total_score:.1f} ({total:.0f}s)')

In [ ]:
final = {}       # fname -> raw bytes
final_costs = {} # fname -> cost

# 1. Start with floor solutions
for fname, raw in floor_solutions.items():
    tid = int(re.match(r'task(\d{3})\.onnx', fname).group(1)) if re.match(r'task(\d{3})\.onnx', fname) else -1
    if tid in EXCLUDED_TASKS:
        continue
    final[fname] = raw
    final_costs[fname] = floor_costs.get(fname, float('inf'))

print(f'Floor tasks: {len(floor_solutions)}')


# 2. Override with our solutions where strictly cheaper — with nu validation
improved = 0
new_solved = 0

for onnx_path in sorted(OUTPUT_DIR.glob('task*.onnx')):
    fname = onnx_path.name
    m_match = re.match(r'task(\d{3})\.onnx', fname)
    if not m_match:
        continue

    task_id_int = int(m_match.group(1))
    raw = onnx_path.read_bytes()

    try:
        m = onnx.load_model_from_string(raw)
        cost = estimate_model_cost(m)
    except Exception:
        continue

    if False and NU_AVAILABLE:
        try:
            sess = ort.InferenceSession(raw, providers=['CPUExecutionProvider'])
            examples = nu.load_examples(task_id_int)
            agi_pass, agi_fail, _ = nu.verify_subset(sess, examples['train'] + examples['test'])
            gen_pass, gen_fail, _ = nu.verify_subset(sess, examples['arc-gen'])
            del sess
            gc.collect()

            if agi_fail > 0 or gen_fail > 0:
                print(f'  SKIP {fname}: nu fail agi={agi_fail} gen={gen_fail}')
                continue
        except Exception as e:
            print(f'  SKIP {fname}: nu error {e}')
            continue

    if fname in final:
        if cost < final_costs[fname]:
            old_score = max(1.0, 25.0 - math.log(max(1, final_costs[fname])))
            new_score = max(1.0, 25.0 - math.log(max(1, cost)))
            print(f'  IMPROVED {fname}: cost {final_costs[fname]}→{cost} (+{new_score-old_score:.2f} pts)')
            final[fname] = raw
            final_costs[fname] = cost
            improved += 1
    else:
        final[fname] = raw
        final_costs[fname] = cost
        score = max(1.0, 25.0 - math.log(max(1, cost)))
        print(f'  NEW {fname}: cost={cost}, score={score:.2f}')
        new_solved += 1


print(f'\nFloor tasks:     {len(floor_solutions)}')
print(f'New tasks added: {new_solved}')
print(f'Tasks improved:  {improved}')
print(f'Total in final:  {len(final)}')


# 3. Re-validate ALL final models — try all candidates, keep cheapest passing one
if False and NU_AVAILABLE:
    print('\nRe-validating all final models — picking cheapest valid candidate per task...')

    to_remove = []
    upgraded = 0

    for fname, raw in sorted(final.items()):
        m = re.match(r'task(\d{3})\.onnx', fname)
        if not m:
            continue

        tid = int(m.group(1))
        if tid in EXCLUDED_TASKS:
            continue

        all_cands = [(raw, 'primary')] + [(br, f'backup{i}') for i, br in enumerate(floor_backups.get(fname, []))]
        best_cost_r = float('inf')
        best_raw_r = None

        for cand_raw, cand_label in all_cands:
            cand_result = validate_official(tid, cand_raw)
            if cand_result is not None and cand_result['cost'] < best_cost_r:
                best_cost_r = cand_result['cost']
                best_raw_r = cand_raw
            if cand_label == 'primary' and best_raw_r is not None:
                break

        if best_raw_r is not None:
            if best_raw_r is not raw:
                print(f'  UPGRADED {fname}: cheaper candidate found (cost={best_cost_r})')
                upgraded += 1

            final[fname] = best_raw_r
            final_costs[fname] = best_cost_r
        else:
            print(f'  FAILED re-validation: {fname} — dropping')
            task_id = fname.replace('.onnx', '')
            if task_id in {'task096', 'task101', 'task133', 'task178', 'task185', 'task234'}:
                print(f"  GAMBLE {fname} — keeping despite validation failure (op_16 solves this)")
            else:
                to_remove.append(fname)

    for fname in to_remove:
        del final[fname]
        del final_costs[fname]

    print(f'Removed {len(to_remove)} invalid models after re-validation.')
    print(f'Upgraded {upgraded} tasks to cheaper candidate.')
    print(f'Final valid count: {len(final)}')

else:
    print('\nnu not available — skipping re-validation (submission may contain invalid models)')


# 4. Dim-scrub safe tasks
scrub_count = 0
for fname in list(final.keys()):
    m_t = re.match(r'task(\d{3})\.onnx', fname)
    if not m_t:
        continue

    tid = int(m_t.group(1))

    if tid in BAD_SCRUB_TASKS:
        continue
    if tid not in SAFE_SCRUB_TASKS:
        continue

    final[fname] = dim_scrub(final[fname])
    scrub_count += 1

print(f'Dim-scrub applied to {scrub_count} tasks.')


# 5. Trusted override (train+test only, no arc-gen)
afr1ste_5501_models = {}
for fname, cands in all_source_models.items():
    for lbl, raw in cands:
        if lbl == 'afr1ste_5501':
            afr1ste_5501_models[fname] = raw
            break

TRUSTED_LABELS = ['konbu17_v117', 'beicicc_6233', 'artem_logic4', 'afr1ste_5501', 'jonathanchan', 'beicicc_5546', 'artem_logic2']

if False and NU_AVAILABLE:
    override_count = 0

    for trusted_label in TRUSTED_LABELS:
        trusted_models = {}

        for fname, cands in all_source_models.items():
            for lbl, raw in cands:
                if lbl == trusted_label:
                    trusted_models[fname] = raw
                    break

        for fname, trusted_raw in trusted_models.items():
            if not _static_ok(trusted_raw):
                continue

            m_t = re.match(r'task(\d{3})\.onnx', fname)
            if not m_t:
                continue

            tid = int(m_t.group(1))

            try:
                sess = ort.InferenceSession(trusted_raw, providers=['CPUExecutionProvider'])
                examples = nu.load_examples(tid)

                agi_pass, agi_fail, _ = nu.verify_subset(
                    sess, examples['train'] + examples['test']
                )

                if agi_fail > 0:
                    continue

                tmp = f'/tmp/trusted{tid:03d}.onnx'
                with open(tmp, 'wb') as f:
                    f.write(trusted_raw)

                macs, mem, params = nu.score_network(tmp)
                if None in (macs, mem, params):
                    continue

                trusted_cost = int(macs + mem + params)

            except Exception:
                continue

            if trusted_cost < final_costs.get(fname, float('inf')):
                final[fname] = trusted_raw
                final_costs[fname] = trusted_cost
                override_count += 1

    print(f'Trusted override: {override_count} tasks replaced.')


# 6. artem_update lock — PRIMARY artifact, post-April-28 metric
import zipfile as _zf

_artifact_dir = Path('/kaggle/input/notebooks/artemnazemtsev/neurogolf-may-1-logic-driven-ensembling-part-4')
_artifact_models = {}
_artifact_loaded = 0

if _artifact_dir.exists():
    _zips = sorted(_artifact_dir.glob('**/*.zip'), key=lambda p: p.stat().st_size, reverse=True)
    if _zips:
        with _zf.ZipFile(_zips[0]) as _z:
            for _name in _z.namelist():
                if _name.endswith('.onnx'):
                    _artifact_models[Path(_name).name] = _z.read(_name)
    else:
        for _f in sorted(_artifact_dir.rglob('task*.onnx')):
            _artifact_models[_f.name] = _f.read_bytes()

_SKIP_ARTEM = set()

if _artifact_models:
    for fname, raw in _artifact_models.items():
        if fname in _SKIP_ARTEM:
            continue
        if _static_ok(raw):
            final[fname] = raw
            _artifact_loaded += 1
    print(f'artem_update lock: {_artifact_loaded} tasks locked (new metric floor ~6638)')
else:
    print('WARNING: artem_update not found — no artifact lock applied')



# Post-artifact blend: konbu17_update wins where file is smaller  [SUBMISSION 2 ONLY]
_konbu_dir = Path('/kaggle/input/notebooks/konbu17/neurogolf-2026-4-30-update-not-using-exploit')
_blend2_count = 0

if _konbu_dir.exists():
    _konbu_zips = sorted(_konbu_dir.glob('**/*.zip'), key=lambda p: p.stat().st_size, reverse=True)
    if _konbu_zips:
        with _zf.ZipFile(_konbu_zips[0]) as _z:
            for _name in _z.namelist():
                if _name.endswith('.onnx'):
                    _fname = Path(_name).name
                    _raw = _z.read(_name)
                    if _fname in final and _static_ok(_raw) and len(_raw) < len(final[_fname]):
                        final[_fname] = _raw
                        _blend2_count += 1

print(f'konbu17_update blend: {_blend2_count} tasks replaced')


# Per-task LB-verified overrides
OVERRIDE_TASKS = {
    'tasktask173.onnx': 'jonathanchan_v3',  # typo, never fires
}

override_forced = 0
for fname, override_label in OVERRIDE_TASKS.items():
    for lbl, raw in all_source_models.get(fname, []):
        if lbl == override_label and _static_ok(raw):
            final[fname] = raw
            print(f'Override: {fname} → {override_label} ({len(raw):,} bytes)')
            override_forced += 1
            break

print(f'Per-task overrides applied: {override_forced}')


# Final shape filter — remove any dynamic-shape models before submission
_shape_removed = []

for fname in list(final.keys()):
    if not _static_ok(final[fname]):
        _shape_removed.append(fname)
        del final[fname]
        final_costs.pop(fname, None)

if _shape_removed:
    print(f'Shape filter removed {len(_shape_removed)} models: {_shape_removed}')
else:
    print('Shape filter: all models OK')


# Force-remove cost=0 tasks — crashes grader with math.log(0)
_ZERO_COST = set()

for fname in _ZERO_COST:
    final.pop(fname, None)
    final_costs.pop(fname, None)

print(f'Removed zero-cost tasks: {[f for f in _ZERO_COST]}')


# 7. Score estimate
solved_score = sum(max(1.0, 25.0 - math.log(max(1, c))) for c in final_costs.values())
unsolved_score = (NUM_TASKS - len(final)) * 1.0
total_est = solved_score + unsolved_score

print(f'\nEstimated score: {total_est:.1f}  (baseline: 2127)')

cost_list = sorted(final_costs.items(), key=lambda x: x[1], reverse=True)

print('\n=== TOP 40 MOST EXPENSIVE TASKS ===')
total_pts = 0
for fname, cost in cost_list[:40]:
    pts = max(1.0, 25.0 - math.log(max(1, cost)))
    total_pts += pts
    print(f'  {fname}: cost={cost:>10,}  pts={pts:.3f}')
print(f'  (top-40 contributes {total_pts:.1f} pts)')

bands = [
    (0, 100, '> 21.9'),
    (100, 1000, '19.8-21.9'),
    (1000, 10000, '14.3-18.1'),
    (10000, 100000, '8.5-14.3'),
    (100000, 10**9, '< 8.5')
]

print('\n=== COST BAND DISTRIBUTION ===')
for lo, hi, label in bands:
    count = sum(1 for c in final_costs.values() if lo <= c < hi)
    print(f'  cost {lo:>8,}-{hi:>10,}  ({label} pts): {count} tasks')

print('\n=== TOP-10 EXPENSIVE TASKS — ALL SOURCE FILE SIZES ===')
top10_fnames = [f for f, _ in cost_list[:10]]

for fname in top10_fnames:
    alts = [(label, len(raw)) for label, raw in all_source_models.get(fname, [])]
    alts.sort(key=lambda x: x[1])

    cur_cost = final_costs[fname]
    cur_size = len(final[fname])

    print(f'\n{fname}  local_onnx_cost={cur_cost:,}  cur_bytes={cur_size:,}')

    for label, fsize in alts[:10]:
        used = ' ← CURRENT' if fsize == cur_size else ''
        print(f'  {label:25s}: {fsize:>8,} bytes{used}')

print(f'Expected gain:   +{total_est - 2127:.1f} pts')


# 8. Write zip
zip_path = OUTPUT_DIR / 'submission.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname, data in sorted(final.items()):
        zf.writestr(fname, data)

size_mb = zip_path.stat().st_size / 1e6
print(f'\nWrote {zip_path} ({size_mb:.1f} MB, {len(final)} tasks)')


# 9. Sanity check
errors = []
with zipfile.ZipFile(zip_path) as zf:
    for info in zf.infolist():
        try:
            onnx.load_model_from_string(zf.read(info.filename))
        except Exception as e:
            errors.append(f'{info.filename}: {e}')

if errors:
    print(f'ERRORS: {errors[:5]}')
else:
    print(f'All {len(final)} ONNX files parse cleanly. Ready to submit!')